# Imports + Data

In [ ]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 6.9 MB/s eta 0:00:00


In [ ]:
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

In [ ]:
!pip install json5

In [ ]:
from huggingface_hub import login
# from llama_cpp import Llama # Works with GPU
import torch
import os
import re
import ast
import time
import pandas as pd
import numpy as np
import openai
from openai import OpenAI
from time import sleep
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
import math
import random
from google import genai
from google.genai import types
import tiktoken
from anthropic import Anthropic
from requests.exceptions import ConnectionError, Timeout, RequestException
from google.api_core import exceptions as gcore_exceptions
import json
import json5
from collections import Counter
from typing import Callable, Optional

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Load CSV file
folder_path = ""
df = pd.read_csv(folder_path+"rotexamples_07072025.csv")

In [ ]:
# Set up API keys
gpt_key = ""
claude_key = ""
gemini_key = ""
hf_key = ""

In [ ]:
# login(hf_key)

# Survey Structure

Moral Emotions: https://pages.stern.nyu.edu/~jhaidt/articles/alternate_versions/haidt.2003.the-moral-emotions.pub025-as-html.html

## Choices

In [ ]:
# Define choice lists with numeric anchors
EMOTIONS = [
    "Contempt", "Anger", "Disgust",
    "Shame", "Embarrassment", "Guilt",
    "Compassion", "Pride",
    "Other", "No emotion"
]

STRENGTHS = [
    "1 = A little", "2 = Moderate", "3 = A lot", "4 = Extremely"
]

ACTIONS = [
    "1 = Do nothing",
    "2 = Stay away or keep a distance from the actor",
    "3 = Gossip about / discuss with / complain to others informally",
    "4 = Verbal reprimand / verbal confrontation",
    "5 = Physical confrontation",
    "6 = Inform the authorities, such as the police",
    "7 = Support the actor",
    "8 = Celebrate / praise"
]

APPROPRIATENESS = [
    "1 = Not at all inappropriate",
    "2 = Somewhat inappropriate",
    "3 = Inappropriate",
    "4 = Very inappropriate",
    "5 = Extremely inappropriate"
]

WRONGNESS = [
    "1 = Not at all wrong",
    "2 = Somewhat wrong",
    "3 = Wrong",
    "4 = Very wrong",
    "5 = Extremely wrong"
]

Definitions below are sourced from [APA Dictionary](https://dictionary.apa.org).

In [ ]:
# Short definitions
MORAL_EMOTIONS_DEFS = """
Available emotions with definitions:
- Contempt: an emotion characterized by negative regard for anything or anybody considered to be inferior, vile, or worthless.
- Anger: an emotion characterized by tension and hostility arising from frustration, real or imagined injury by another, or perceived injustice.
- Disgust: a strong aversion, for example, to the taste, smell, or touch of something deemed revolting, or toward a person or behavior deemed morally repugnant.
- Shame: a highly unpleasant self-conscious emotion arising from the sense of there being something dishonorable, immodest, or indecorous in one’s own conduct or circumstances.
- Embarrassment: a self-conscious emotion in which a person feels awkward or flustered in other people’s company or because of the attention of others, as, for example, when being observed engaging in actions that are subject to mild disapproval from others.
- Guilt: a self-conscious emotion characterized by a painful appraisal of having done (or thought) something that is wrong and often by a readiness to take action designed to undo or mitigate this wrong.
- Compassion: a strong feeling of sympathy with another person’s feelings of sorrow or distress, usually involving a desire to help or comfort that person.
- Pride: a self-conscious emotion that occurs when a goal has been attained and one’s achievement has been recognized and approved by others.
- Other: any emotion not listed above.
- No emotion: indicates absence of emotional response.
"""

## Questions

In [ ]:
QUESTION_TEXTS = {
    # Actor
    1: (
        "1) How would {actor} feel about their action? "
        "(select emotions from the list; if not listed, add them under 'other_emotions'; "
        "if 'No emotion', it must be the ONLY choice): "
        + ", ".join(EMOTIONS[:-1])
        + "\n\n" + MORAL_EMOTIONS_DEFS
    ),
    2: "2) For each selected emotion (including any in 'other_emotions'), how strongly would {actor} feel it? (choose one; if 'No emotion' was selected in Q1, answer = null/NaN): " + ", ".join(STRENGTHS),

    # Close friends or relatives
    3: (
        "3) Suppose {actor}'s close friends or relatives observed the action. "
        "How would they feel? (select emotions from the list; if not listed, add them under 'other_emotions'; "
        "if 'No emotion', it must be the ONLY choice): "
        + ", ".join(EMOTIONS[:-1])
        + "\n\n" + MORAL_EMOTIONS_DEFS
    ),
    4: "4) How strongly would close friends or relatives feel each selected emotion (including any in 'other_emotions')? (choose one; if 'No emotion' was selected in Q3, answer = null/NaN): " + ", ".join(STRENGTHS),
    5: "5) How would close friends or relatives react toward {actor}'s action? (choose one): " + ", ".join(ACTIONS),
    6: "6) How should close friends or relatives react toward {actor}'s action? (choose one): " + ", ".join(ACTIONS),

    # Acquaintances
    7: (
        "7) Suppose {actor}'s acquaintances observed the action. How would they feel? "
        "(select emotions from the list; if not listed, add them under 'other_emotions'; "
        "if 'No emotion', it must be the ONLY choice): "
        + ", ".join(EMOTIONS[:-1])
        + "\n\n" + MORAL_EMOTIONS_DEFS
    ),
    8: "8) How strongly would acquaintances feel each selected emotion (including any in 'other_emotions')? (choose one; if 'No emotion' was selected in Q7, answer = null/NaN): " + ", ".join(STRENGTHS),
    9: "9) How would acquaintances react toward {actor}'s action? (choose one): " + ", ".join(ACTIONS),
    10: "10) How should acquaintances react toward {actor}'s action? (choose one): " + ", ".join(ACTIONS),

    # Strangers
    11: (
        "11) Suppose strangers observed the action. How would they feel? "
        "(select emotions from the list; if not listed, add them under 'other_emotions'; "
        "if 'No emotion', it must be the ONLY choice): "
        + ", ".join(EMOTIONS[:-1])
        + "\n\n" + MORAL_EMOTIONS_DEFS
    ),
    12: "12) How strongly would strangers feel each selected emotion (including any in 'other_emotions')? (choose one; if 'No emotion' was selected in Q11, answer = null/NaN): " + ", ".join(STRENGTHS),
    13: "13) How would strangers react toward {actor}'s action? (choose one): " + ", ".join(ACTIONS),
    14: "14) How should strangers react toward {actor}'s action? (choose one): " + ", ".join(ACTIONS),

    # Norm judgments
    15: "15) To what extent do you think {actor}'s action is inappropriate? (choose one): " + ", ".join(APPROPRIATENESS),
    16: "16) To what extent do you think {actor}'s action is wrong? (choose one): " + ", ".join(WRONGNESS),
}

In [ ]:
# Descriptive names for questions
QUESTION_COLUMNS = {
    # Actor emotions
    1: "actor_emotions",
    2: "actor_emotion_strength",

    # Close friends or relatives
    3: "friends_emotions",
    4: "friends_emotion_strength",
    5: "friends_reaction",
    6: "friends_reaction_should",

    # Acquaintances
    7: "acquaintances_emotions",
    8: "acquaintances_emotion_strength",
    9: "acquaintances_reaction",
    10: "acquaintances_reaction_should",

    # Strangers
    11: "strangers_emotions",
    12: "strangers_emotion_strength",
    13: "strangers_reaction",
    14: "strangers_reaction_should",

    # Norm judgments
    15: "norm_inappropriateness",
    16: "norm_wrongness"
}

In [ ]:
def rename_question_columns(df):
    """Rename Q1..Q1k to descriptive names."""
    rename_map = {f"Q{k}": v for k, v in QUESTION_COLUMNS.items()}
    return df.rename(columns=rename_map)

## Pre-defined schema

In [ ]:
# Pre-defined JSON schema
def build_question_schema(actor):
    questionnaire_schema = {
        "name": "norm_questionnaire",
        "schema": {
            "type": "object",
            "properties": {
                # Actor emotions
                "Q1": {
                    "type": "object",
                    "description": QUESTION_TEXTS[1].format(actor=actor),
                    "properties": {
                        "emotions": {
                            "type": "array",
                            "items": {"type": "string", "enum": EMOTIONS[:-2]},  # exclude Other + No emotion
                            "uniqueItems": True
                        },
                        "other_emotions": {
                            "type": "array",
                            "items": {
                                "type": "string",
                                "not": {"enum": EMOTIONS}  # cannot be any known emotion
                            },
                            "uniqueItems": True
                        },
                        "no_emotion": {
                            "type": "boolean",
                            "description": "True if no emotion applies; if true, both 'emotions' and 'other_emotions' must be empty."
                        }
                    },
                    "additionalProperties": False
                },
                "Q2": {
                    "type": ["object", "null"],
                    "description": QUESTION_TEXTS[2].format(actor=actor),
                    "patternProperties": {
                        ".*": {"type": "integer", "enum": [1, 2, 3, 4]}
                    }
                },

                # --- Friends ---
                "Q3": {
                    "type": "object",
                    "description": QUESTION_TEXTS[3].format(actor=actor),
                    "properties": {
                        "emotions": {
                            "type": "array",
                            "items": {"type": "string", "enum": EMOTIONS[:-2]},
                            "uniqueItems": True
                        },
                        "other_emotions": {
                            "type": "array",
                            "items": {
                                "type": "string",
                                "not": {"enum": EMOTIONS}
                            },
                            "uniqueItems": True
                        },
                        "no_emotion": {"type": "boolean"}
                    },
                    "additionalProperties": False
                },
                "Q4": {
                    "type": ["object", "null"],
                    "description": QUESTION_TEXTS[4],
                    "patternProperties": {
                        ".*": {"type": "integer", "enum": [1, 2, 3, 4]}
                    }
                },
                "Q5": {
                    "type": "integer",
                    "description": QUESTION_TEXTS[5].format(actor=actor),
                    "enum": [1, 2, 3, 4, 5, 6, 7, 8]
                },
                "Q6": {
                    "type": "integer",
                    "description": QUESTION_TEXTS[6],
                    "enum": [1, 2, 3, 4, 5, 6, 7, 8]
                },

                # Acquaintances
                "Q7": {
                    "type": "object",
                    "description": QUESTION_TEXTS[7].format(actor=actor),
                    "properties": {
                        "emotions": {
                            "type": "array",
                            "items": {"type": "string", "enum": EMOTIONS[:-2]},
                            "uniqueItems": True
                        },
                        "other_emotions": {
                            "type": "array",
                            "items": {
                                "type": "string",
                                "not": {"enum": EMOTIONS}
                            },
                            "uniqueItems": True
                        },
                        "no_emotion": {"type": "boolean"}
                    },
                    "additionalProperties": False
                },
                "Q8": {
                    "type": ["object", "null"],
                    "description": QUESTION_TEXTS[8],
                    "patternProperties": {
                        ".*": {"type": "integer", "enum": [1, 2, 3, 4]}
                    }
                },
                "Q9": {
                    "type": "integer",
                    "description": QUESTION_TEXTS[9].format(actor=actor),
                    "enum": [1, 2, 3, 4, 5, 6, 7, 8]
                },
                "Q10": {
                    "type": "integer",
                    "description": QUESTION_TEXTS[10],
                    "enum": [1, 2, 3, 4, 5, 6, 7, 8]
                },

                # Strangers
                "Q11": {
                    "type": "object",
                    "description": QUESTION_TEXTS[11].format(actor=actor),
                    "properties": {
                        "emotions": {
                            "type": "array",
                            "items": {"type": "string", "enum": EMOTIONS[:-2]},
                            "uniqueItems": True
                        },
                        "other_emotions": {
                            "type": "array",
                            "items": {
                                "type": "string",
                                "not": {"enum": EMOTIONS}
                            },
                            "uniqueItems": True
                        },
                        "no_emotion": {"type": "boolean"}
                    },
                    "additionalProperties": False
                },
                "Q12": {
                    "type": ["object", "null"],
                    "description": QUESTION_TEXTS[12],
                    "patternProperties": {
                        ".*": {"type": "integer", "enum": [1, 2, 3, 4]}
                    }
                },
                "Q13": {
                    "type": "integer",
                    "description": QUESTION_TEXTS[13].format(actor=actor),
                    "enum": [1, 2, 3, 4, 5, 6, 7, 8]
                },
                "Q14": {
                    "type": "integer",
                    "description": QUESTION_TEXTS[14],
                    "enum": [1, 2, 3, 4, 5, 6, 7, 8]
                },

                # Norm judgments
                "Q15": {
                    "type": "integer",
                    "description": QUESTION_TEXTS[15].format(actor=actor),
                    "enum": [1, 2, 3, 4, 5]
                },
                "Q16": {
                    "type": "integer",
                    "description": QUESTION_TEXTS[16].format(actor=actor),
                    "enum": [1, 2, 3, 4, 5]
                }
            },
            "additionalProperties": False
        }
    }
    return questionnaire_schema

## Prompt

In [ ]:
def build_prompt(stimuli, actor, schema=None, model="gpt"):
    """
    Build the system prompt differently depending on the model.
    For GPT/Claude: no schema injection.
    For Gemini: embed the JSON schema in the instructions.
    """
    base_instructions = f"""
    You are participating in an academic research study on moral judgments.
    The scenario may involve sensitive topics, but they are purely hypothetical and for research purposes only.
    You are given a statement about a behavior:

    \"\"\"{stimuli}\"\"\"

    The actor is: {actor}.

    Your task is to answer a fixed set of 16 questions about this behavior.
    Each question has clearly defined answer choices.

    Instructions:
    - For most questions, use ONLY the provided answer choices.
    - Do not rephrase or alter the provided options.
    - For emotions (Q1, Q3, Q7, Q11):
        * If 'No emotion' is selected, it must be the ONLY choice, and the corresponding intensity (Q2, Q4, Q8, Q12) must be null/NaN.
        * If the actor’s feelings include something not in the provided list, add it under 'other_emotions', NOT 'emotions'.
          (Example: 'Frustration' is not in the list, so it must go into 'other_emotions', NOT 'emotions'.)
        * 'emotions' MUST contain only items from the provided list. DO NOT put any unlisted emotions (e.g., 'Frustration') into 'emotions'.
        * 'other_emotions' must contain only new, unlisted emotions, and never duplicates of those already chosen in 'emotions'.
    - Each intensity question (Q2, Q4, Q8, Q12) must include all emotions selected in the previous question, including any in 'other_emotions'.
    - Do not include commentary, apologies, or explanations. Output ONLY valid JSON.
    - Output ONLY valid JSON that strictly follows the provided schema.
    - Ensure all strings are properly quoted and escaped.
    - End all string values and object properties correctly.
    """

    if model.startswith("gemini"):
        return f"""{base_instructions}
        CRITICAL: You must produce ONLY a valid JSON object that exactly matches this schema. Do not include any text before or after the JSON:

        {json.dumps(schema, indent=2)}

        The JSON must be complete, well-formed, and contain no syntax errors. Every string must be properly quoted and terminated.
        ---
        Now, fill in all answers (Q1–Q16) for the given behavior and actor.
        """
    else:
        return f"""{base_instructions}
        Now, fill in all answers (Q1–Q16) for the given behavior and actor.
        {json.dumps(schema, indent=2)}
        """

## Helpers

In [ ]:
def cleanup_json_string(json_str: str) -> str:
    """
    Cleans up common malformed JSON patterns returned by LLMs so that
    the string can be safely parsed with `json.loads`.
    """
    fixed = json_str

    # { 3 } → 3
    fixed = re.sub(r'\{\s*(\d+)\s*\}', r'\1', fixed)

    # {"3"} → 3
    fixed = re.sub(r'\{\s*"(\d+)"\s*\}', r'\1', fixed)

    # {"value": "3"} → 3
    fixed = re.sub(r'\{\s*"value"\s*:\s*"(\d+)"\s*\}', r'\1', fixed)

    # {"value": 3} → 3
    fixed = re.sub(r'\{\s*"value"\s*:\s*(\d+)\s*\}', r'\1', fixed)

    # {"value": null} or {"value": None} → null
    fixed = re.sub(r'\{\s*"value"\s*:\s*(null|None)\s*\}', 'null', fixed)

    # {"enum": "3"} → 3
    fixed = re.sub(r'\{\s*"enum"\s*:\s*"(\d+)"\s*\}', r'\1', fixed)

    # {"3": 1} → 1
    fixed = re.sub(r'\{\s*"\d+"\s*:\s*(\d+)\s*\}', r'\1', fixed)

    # Numbers as strings → numbers: "7" → 7
    fixed = re.sub(r'"\s*(\d+)\s*"', r'\1', fixed)

    # Remove trailing commas before } or ]
    fixed = re.sub(r',(\s*[}\]])', r'\1', fixed)

    # Wrap unquoted keys (like Q1: → "Q1":)
    fixed = re.sub(r'([{,]\s*)([A-Za-z_][A-Za-z0-9_]*)(\s*:)', r'\1"\2"\3', fixed)

    # Strip code fences if Claude wrapped the JSON
    fixed = re.sub(r"^```(?:json)?\s*", "", fixed.strip(), flags=re.IGNORECASE)
    fixed = re.sub(r"\s*```$", "", fixed)

    return fixed

In [ ]:
def normalize_emotions(row, allowed_emotions, use_words=False):
    """
    Normalizes emotion responses for Q1/Q3/Q7/Q11 or their word-based equivalents.
    - Moves unlisted emotions into 'other_emotions'.
    - Ensures 'emotions' only contains allowed items.
    - If 'no_emotion' is True or lists are empty:
        → sets the matching intensity column to None
        → clears 'emotions' and 'other_emotions'.
    - If 'other_emotions' contains allowed items, they are moved back into 'emotions'.
    """
    # Define pairs of (emotion_col, intensity_col)
    if use_words:
        pairs = [
            ("actor_emotions", "actor_emotion_strength"),
            ("friends_emotions", "friends_emotion_strength"),
            ("acquaintances_emotions", "acquaintances_emotion_strength"),
            ("strangers_emotions", "strangers_emotion_strength"),
        ]
    else:
        pairs = [
            ("Q1", "Q2"),
            ("Q3", "Q4"),
            ("Q7", "Q8"),
            ("Q11", "Q12"),
        ]

    for emo_col, strength_col in pairs:
        emo = row[emo_col]
        if isinstance(emo, dict):
            # Normalize to lists
            base_emotions = emo.get("emotions", [])
            if not isinstance(base_emotions, list):
                base_emotions = list(base_emotions)

            extra_other = emo.get("other_emotions", [])
            if not isinstance(extra_other, list):
                extra_other = list(extra_other)

            # Split into valid vs other
            valid = [e for e in base_emotions if e in allowed_emotions]
            other = [e for e in base_emotions if e not in allowed_emotions] + list(extra_other)

            # Move any allowed items from other → emotions
            still_other = []
            for e in other:
                if e in allowed_emotions:
                    valid.append(e)
                else:
                    still_other.append(e)

            emo["emotions"] = valid
            emo["other_emotions"] = still_other

            # Consistency: if no_emotion=True OR lists empty → wipe everything
            if emo.get("no_emotion", False) or (not emo["emotions"] and not emo["other_emotions"]):
                row[strength_col] = None
                emo["emotions"] = []
                emo["other_emotions"] = []
                emo["no_emotion"] = True  # Enforce consistency

            row[emo_col] = emo
    return row

In [ ]:
def post_process_response(response):
    """
    Post-process the response to fix common issues with Gemma output
    """
    processed = {}

    for key, value in response.items():
        if key in ["Q5", "Q6", "Q9", "Q10", "Q13", "Q14", "Q15", "Q16"]:
            # These should be single integer values, not objects
            if isinstance(value, dict):
                # Find the non-null value in the object
                for k, v in value.items():
                    if v is not None:
                        try:
                            processed[key] = int(k)
                            break
                        except ValueError:
                            processed[key] = None
                else:
                    processed[key] = None
            else:
                processed[key] = value
        elif key in ["Q2", "Q4", "Q8", "Q12"]:
            # These should have integer values in the object, not string
            if isinstance(value, dict):
                processed[key] = {k: v for k, v in value.items() if v is not None}
            else:
                processed[key] = value
        else:
            processed[key] = value

    return processed

In [ ]:
def normalize_emotion_strength(val):
    """Convert dict values to int and remove None entries; leave non-dict values unchanged."""
    if isinstance(val, dict):
        return {k: int(v) for k, v in val.items() if v is not None}
    return val  # Leave NaN/None as is

In [ ]:
def clean_strength_columns(df):
    """Remove None values from all *_strength dict columns in DataFrame."""
    strength_cols = [c for c in df.columns if c.endswith("_strength")]

    def clean_dict(d):
        return {k: v for k, v in d.items() if v is not None} if isinstance(d, dict) else d

    for col in strength_cols:
        df[col] = df[col].apply(clean_dict)
    return df

In [ ]:
def collect_unique_emotions(df):
    """
    Extracts and collects unique emotions from a DataFrame
    with columns ['actor_emotions', 'friends_emotions', 'acquaintances_emotions', 'strangers_emotions'].

    Parameters:
        df (pd.DataFrame): Input DataFrame containing emotion dicts or strings of dicts.

    Returns:
        unique_emotions (dict): Mapping of category -> sorted list of unique emotions
        all_unique (list): Sorted list of all unique emotions across categories
    """

    def extract_emotions(cell):
        """Helper: extract emotions + other_emotions from dict-like cells."""
        if pd.isna(cell):
            return []
        if isinstance(cell, str):
            try:
                cell = ast.literal_eval(cell)  # convert string-dict to real dict
            except Exception:
                return []
        if not isinstance(cell, dict):
            return []
        emotions = cell.get("emotions", [])
        other = cell.get("other_emotions", [])
        return emotions + other

    # Collect unique emotions per category
    unique_emotions = {}
    col_map = {
        "actor_emotions": "self",
        "friends_emotions": "friends",
        "acquaintances_emotions": "acquaintances",
        "strangers_emotions": "strangers"
    }

    for col, label in col_map.items():
        if col in df.columns:
            all_emotions = df[col].apply(extract_emotions).explode().dropna().unique()
            unique_emotions[label] = sorted(set(all_emotions))
        else:
            unique_emotions[label] = []

    # Combine across all categories
    all_unique = sorted(set(
        emotion for emotions in unique_emotions.values() for emotion in emotions
    ))

    # Print results
    for k, v in unique_emotions.items():
        print(f"{k}: {v}")
    print(all_unique)
    print(len(all_unique), "unique emotions total")

    return unique_emotions, all_unique

In [ ]:
def print_emotion_strength_counts(df):
    """
    Takes a dataframe with *_emotion_strength columns containing dicts of {emotion: strength}.
    Prints how many times each emotion appears across rows for each column.
    """
    # Columns that contain the emotion-strength dicts
    strength_cols = [
        "actor_emotion_strength",
        "friends_emotion_strength",
        "acquaintances_emotion_strength",
        "strangers_emotion_strength"
    ]

    # Ensure dict format (convert strings if needed)
    for col in strength_cols:
        if col in df.columns:
            df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

    # Count occurrences of each emotion
    emotion_counts = {}
    for col in strength_cols:
        if col in df.columns:
            counter = Counter()
            for row in df[col].dropna():
                counter.update(row.keys())
            emotion_counts[col] = dict(counter)

    # Print results
    for col, counts in emotion_counts.items():
        print(f"\nCounts for {col}:")
        for emotion, count in sorted(counts.items(), key=lambda x: (-x[1], x[0])):
            print(f"  {emotion}: {count}")

In [ ]:
def build_reaction_counts(df, model='gpt'):
    """
    Build a reaction counts matrix (values 1–8 vs reaction variables)
    and save it as a CSV.

    Parameters:
        df (pd.DataFrame): Input dataframe with reaction columns.
        output_file (str): Path to save the output CSV (default: 'reaction_counts.csv').

    Returns:
        pd.DataFrame: The reaction counts matrix.
    """
    # Variables to analyze
    cols = [
        'friends_reaction',
        'friends_reaction_should',
        'acquaintances_reaction',
        'acquaintances_reaction_should',
        'strangers_reaction',
        'strangers_reaction_should'
    ]

    # Initialize result DataFrame
    matrix = pd.DataFrame(index=range(1, 9), columns=cols).fillna(0)

    # Fill counts
    for col in cols:
        counts = df[col].value_counts()
        for val, count in counts.items():
            if val in matrix.index:
                matrix.loc[val, col] = count

    matrix.to_csv(f'{model}_reaction_counts.csv', index_label="Value")
    return matrix

# Major Pipeline with LLMs

In [ ]:
def with_retries(fn, *, backend="gpt", max_tries=None, base=None, jitter=None):
    """
    Exponential backoff with jitter for transient/API errors.

    Args:
        fn (callable): Function to execute.
        backend (str): "gemini", "claude", or "gpt".
        max_tries (int): Max attempts (default depends on backend).
        base (float): Base delay (default depends on backend).
        jitter (float): Random jitter added (default depends on backend).

    Returns:
        Any: Result of fn() if successful, otherwise raises last exception.
    """
    # Backend-specific defaults
    defaults = {
        "gemini": {"max_tries": 8, "base": 1.0, "jitter": 1.0},
        "claude": {"max_tries": 6, "base": 0.75, "jitter": 0.5},
        "gpt":    {"max_tries": 6, "base": 0.75, "jitter": 0.5},
    }
    cfg = defaults.get(backend, defaults["gpt"])  # fallback to GPT defaults

    max_tries = max_tries or cfg["max_tries"]
    base = base or cfg["base"]
    jitter = jitter or cfg["jitter"]

    # Gemini has a richer set of retriable exceptions
    retriable = (
        gcore_exceptions.DeadlineExceeded,
        gcore_exceptions.ServiceUnavailable,
        gcore_exceptions.Aborted,
        gcore_exceptions.ResourceExhausted,
        gcore_exceptions.InternalServerError,
        ConnectionError,
        Timeout,
        RequestException,
        Exception,  # catch-all
    ) if backend == "gemini" else (Exception,)

    # Retry loop
    for attempt in range(1, max_tries + 1):
        try:
            return fn()
        except retriable as e:
            if attempt == max_tries:
                print(f"[{backend.upper()}] All {max_tries} attempts failed: {type(e).__name__}: {e}")
                raise
            delay = base * (2 ** (attempt - 1)) + random.random() * jitter
            print(f"[{backend.upper()}] Attempt {attempt} failed ({type(e).__name__}): {e}\nRetrying in {delay:.2f}s...")
            sleep(delay)

In [ ]:
def run_all(df, backend="gpt", max_workers=1, out_csv=None, out_parquet=None):
    """
    Run all question templates on a DataFrame of stimuli using multithreading.

    For each row in df, generates responses for male and female variants of the stimuli,
    runs the appropriate ask_sequentially_* function (gpt/claude/gemini),
    and collects results into a new DataFrame.

    Args:
        df (pd.DataFrame): Input dataframe with columns
            ["stimuli-male", "norm-target in stimuli-male",
             "stimuli-female", "norm-target in stimuli-female"].
        question_templates (list): List of lambda functions for per-question prompts.
        backend (str): Which LLM backend to use ("gpt", "claude", "gemini").
        max_workers (int): Number of threads to use (parallelism).
        out_csv (str or None): Path to save the output CSV. Defaults to a name based on backend.
        out_parquet (str or None): Path to save the output parquet. Defaults to a name based on backend.

    Returns:
        pd.DataFrame: DataFrame of responses with descriptive column names,
        cleaned of artifacts like "Answer x:".
    """
    tasks = []
    n = len(df)
    results = [None] * (2 * n)

    # Pick backend + default CSV
    if backend == "gpt":
        ask_fn = ask_sequentially_gpt
        default_csv = "gpt_responses.csv"
        default_parquet = "gpt_responses.parquet"
    elif backend == "claude":
        ask_fn = ask_sequentially_claude
        default_csv = "claude_responses.csv"
        default_parquet = "claude_responses.parquet"
    elif backend == "gemini":
        ask_fn = ask_sequentially_gemini
        default_csv = "gemini_responses.csv"
        default_parquet = "gemini_responses.parquet"
    elif backend == "gpt-oss":
        ask_fn = ask_sequentially_gpt_oss
        default_csv = "gpt_oss_responses.csv"
        default_parquet = "gpt_oss_responses.parquet"
        max_workers = 1  # Force single-threaded for safety
    elif backend == "llama":
        ask_fn = ask_sequentially_llama
        default_csv = "llama_responses.csv"
        default_parquet = "llama_responses.parquet"
        max_workers = 1  # Force single-threaded for safety
    elif backend == "gemma":
        ask_fn = ask_sequentially_gemma
        default_csv = "gemma_responses.csv"
        default_parquet = "gemma_responses.parquet"
        max_workers = 1  # Force single-threaded for safety
    else:
        raise ValueError(f"Unsupported backend: {backend}")

    if out_csv is None:
        out_csv = default_csv

    if out_parquet is None:
        out_parquet = default_parquet

    def _one(index, stim, actor, gender, row_id):
      try:
          res = ask_fn(stim, actor)
          res["stimuli"] = stim
          res["actor"] = actor
          res["gender"] = gender
          res["ID"] = row_id   # Keep original ID
          return index, gender, res
      except Exception as e:
          print(f"Error processing {actor} ({gender}): {e}")
          return index, gender, {
              "stimuli": stim,
              "actor": actor,
              "gender": gender,
              "ID": row_id,
              "error": str(e)
          }

    print(f"Processing {n} rows with max_workers={max_workers} using backend={backend}")
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        for idx, row in df.reset_index(drop=True).iterrows():
          stim_m = row["stimuli-male"]
          actor_m = row["norm-target in stimuli-male"]
          stim_f = row["stimuli-female"]
          actor_f = row["norm-target in stimuli-female"]
          row_id = row["ID"]   # <- grab original ID

          print(f"Queued (male):   {stim_m} by {actor_m}")
          print(f"Queued (female): {stim_f} by {actor_f}")

          tasks.append(ex.submit(_one, idx, stim_m, actor_m, "male", row_id))
          tasks.append(ex.submit(_one, idx, stim_f, actor_f, "female", row_id))

        for fut in as_completed(tasks):
            idx, gender, res = fut.result()
            pos = 2 * idx + (0 if gender == "male" else 1)
            results[pos] = res
            print(f"Completed: {res.get('actor', 'unknown')} ({gender})")

    to_df = pd.DataFrame(results)

    front = [c for c in ["ID", "stimuli", "actor", "gender"] if c in to_df.columns]
    to_df = to_df[front + [c for c in to_df.columns if c not in front]]

    # Clean emotions
    to_df = to_df.apply(lambda r: normalize_emotions(r, EMOTIONS), axis=1)

    # Cleanup: remove "Answer x:" artifacts if any
    to_df = to_df.replace(r'Answer\s*\d+\s*:\s*', '', regex=True)

    # Rename Q1..Q22 to descriptive names
    to_df = rename_question_columns(to_df)

    # Normalize reaction, inappropriateness, wrongness columns as numeric
    to_df["friends_reaction"] = pd.to_numeric(to_df["friends_reaction"], errors="coerce")
    to_df["friends_reaction_should"] = pd.to_numeric(to_df["friends_reaction_should"], errors="coerce")
    to_df["acquaintances_reaction"] = pd.to_numeric(to_df["acquaintances_reaction"], errors="coerce")
    to_df["acquaintances_reaction_should"] = pd.to_numeric(to_df["acquaintances_reaction_should"], errors="coerce")
    to_df["strangers_reaction"] = pd.to_numeric(to_df["strangers_reaction"], errors="coerce")
    to_df["strangers_reaction_should"] = pd.to_numeric(to_df["strangers_reaction_should"], errors="coerce")
    to_df["norm_inappropriateness"] = pd.to_numeric(to_df["norm_inappropriateness"], errors="coerce")
    to_df["norm_wrongness"] = pd.to_numeric(to_df["norm_wrongness"], errors="coerce")

    # Normalize emotion stregth columns
    to_df["actor_emotion_strength"] =to_df["actor_emotion_strength"].apply(normalize_emotion_strength)
    to_df["friends_emotion_strength"] =to_df["friends_emotion_strength"].apply(normalize_emotion_strength)
    to_df["acquaintances_emotion_strength"] =to_df["acquaintances_emotion_strength"].apply(normalize_emotion_strength)
    to_df["strangers_emotion_strength"] =to_df["strangers_emotion_strength"].apply(normalize_emotion_strength)

    to_df.to_csv(folder_path+out_csv, index=False)
    print(f"Done: responses saved to {out_csv}")
    to_df.to_parquet(folder_path+out_parquet, index=False)
    print(f"Done: responses saved to {out_parquet}")
    return to_df

## Closed-Source

### GPT

In [ ]:
def ask_sequentially_gpt(stimuli, actor, model="gpt-5.2", response=False):
    from openai import OpenAI
    import json, json5

    client = OpenAI(api_key=gpt_key)

    schema = build_question_schema(actor)
    prompt = build_prompt(stimuli, actor, schema, model="gpt")

    resp = with_retries(
        lambda: client.responses.create(
            model=model,
            reasoning={"effort": "high"},
            input=prompt,
            temperature=1.0,
            max_output_tokens=8192,
        ),
        backend="gpt"
    )

    if response:
        return resp

    text = resp.output_text
    if not text:
        raise ValueError("GPT returned empty output")

    try:
        return json.loads(cleanup_json_string(text))
    except json.JSONDecodeError:
        return json5.loads(text)


### Claude

In [ ]:
# Helpers
def _extract_text(message_response) -> str:
    """Claude returns a list of content blocks; concatenate any text blocks."""
    out = []
    if getattr(message_response, "content", None):
        for part in message_response.content:
            if getattr(part, "type", "") == "text" and getattr(part, "text", None):
                out.append(part.text)
    return "\n".join(out).strip()


# New JSON-schema main function
def ask_sequentially_claude(stimuli, actor, model="claude-opus-4-5-20251101", response=False):
    """
    Runs the full Q1–16 questionnaire for one (stimuli, actor) pair
    in a single call, asking for JSON.
    """
    # Light concurrency / rate limiting
    def make_client():
      return Anthropic(api_key=claude_key)

    client = make_client()
    schema = build_question_schema(actor)
    prompt = build_prompt(stimuli, actor, model='claude')

    resp = with_retries(
        lambda: client.messages.create(
            model=model,
            system=prompt,
            messages=[{"role": "user", "content": json.dumps(schema, indent=2)}],
            temperature=0,
            max_tokens=2048,
        ),
        backend="claude"
    )

    if response:
      return resp

    reply = _extract_text(resp)
    reply_clean = cleanup_json_string(reply)

    try:
        answers = json.loads(reply_clean)
    except json.JSONDecodeError:
        raise ValueError(f"Claude did not return valid JSON:\n{reply}")

    return answers

### Gemini

In [ ]:
# Main function
def ask_sequentially_gemini(stimuli, actor, model_name="gemini-3-pro-preview", response=False):
    """
    One-shot JSON schema run with Gemini.
    No conversation history, returns answers to ALL questions in one JSON object.
    """
    # Client
    def make_client():
        return genai.Client(api_key=gemini_key)

    client = make_client()

    # Build schema & prompt
    schema = build_question_schema(actor)
    prompt = build_prompt(stimuli, actor, schema, model='gemini')

    # Safety settings
    safety_settings = [
        types.SafetySetting(
            category=types.HarmCategory.HARM_CATEGORY_HARASSMENT,
            threshold=types.HarmBlockThreshold.BLOCK_NONE,
        ),
        types.SafetySetting(
            category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
            threshold=types.HarmBlockThreshold.BLOCK_NONE,
        ),
        types.SafetySetting(
            category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
            threshold=types.HarmBlockThreshold.BLOCK_NONE,
        ),
        types.SafetySetting(
            category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
            threshold=types.HarmBlockThreshold.BLOCK_NONE,
        ),
    ]

    config = types.GenerateContentConfig(
        temperature=0.0,
        max_output_tokens=20000,
        top_p=1.0,
        safety_settings=safety_settings,
        response_mime_type="application/json",
    )

    def _call():
        return client.models.generate_content(
            model=model_name,
            contents=prompt,
            config=config,
        )

    resp = with_retries(_call, backend="gemini")

    if response:
        return resp

    try:
        answer_json = resp.candidates[0].content.parts[0].text
        reply_clean = cleanup_json_string(answer_json)
        answers = json.loads(reply_clean)
    except Exception as e:
        print(resp)
        print(f"[Gemini] Failed to parse response: {e}")
        answers = {}

    return answers

## Open-Source

### GPT-oss

In [ ]:
llm = Llama.from_pretrained(
	repo_id="unsloth/gpt-oss-20b-GGUF",
	filename="gpt-oss-20b-F16.gguf",
  n_ctx=8192,        # Set depending on context window needed
  n_gpu_layers=-1,   # Put all layers on GPU
  verbose=False,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


./gpt-oss-20b-F16.gguf:   0%|          | 0.00/13.8G [00:00<?, ?B/s]

llama_context: n_ctx_seq (8192) < n_ctx_train (131072) -- the full capacity of the model will not be utilized
llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


In [ ]:
def ask_sequentially_gpt_oss(stimuli, actor, max_new_tokens=4112):
    """
    Runs the full Q1–Q16 questionnaire for one (stimuli, actor) pair
    using gpt-oss. Returns a dict with Q1–Q16 answers.
    """
    schema = build_question_schema(actor)
    schema_text = json.dumps(schema, indent=2)

    # Build prompt with schema injection
    prompt = f"""You are participating in an academic research study on moral judgments. The scenario may involve sensitive topics, but they are purely hypothetical and for research purposes only.

You are given a statement about a behavior:

\"\"\"{stimuli}\"\"\"

The actor is: {actor}.

Your task is to answer a fixed set of 16 questions (Q1–Q16) about this behavior.

OUTPUT RULES:
- Output ONLY valid JSON.
- Do not include explanations, reasoning, or commentary.
- Begin with '{{' and end with '}}'.
- Ensure ALL keys "Q1" through "Q16" are present.
- For Q1, Q3, Q7, Q11 → return an object with keys "emotions", "other_emotions", "no_emotion".
- For Q2, Q4, Q8, Q12 → return an object with emotion names as keys and integer ratings as values.
- For Q5, Q6, Q9, Q10, Q13, Q14, Q15, Q16 → return a single integer.

Schema:
{schema_text}

Now, fill in all answers (Q1–Q16) for the given behavior and actor.
"""

    # Run inference
    output = llm(
        prompt,
        temperature=1.0,
        max_tokens=max_new_tokens,
        # Remove problematic stop token
        echo=False,
    )

    reply_text = output["choices"][0]["text"].strip()

    # Extract JSON substring
    # Look for the LAST complete JSON object, not the first opening brace
    lines = reply_text.split('\n')
    json_str = None

    # Try to find a line that looks like complete JSON starting with {
    for line in reversed(lines):
        line = line.strip()
        if line.startswith('{') and line.endswith('}'):
            json_str = line
            break

    # Fallback: look for JSON in the last part of the text
    if json_str is None:
        # Find the last occurrence of a complete JSON object pattern
        import re
        json_pattern = r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}(?:\s*$)'
        matches = list(re.finditer(json_pattern, reply_text, re.DOTALL))
        if matches:
            json_str = matches[-1].group().strip()

    if json_str is None:
        print(reply_text)
        print("ERROR: No complete JSON object found in output")
        return {f"Q{i}": None for i in range(1, 17)}

    # Cleanup common issues
    try:
        fixed = cleanup_json_string(json_str)
        parsed = json.loads(fixed)

    except json.JSONDecodeError as e:
        print(reply_text)
        print(f"JSON DECODE ERROR: {e}")
        return {f"Q{i}": None for i in range(1, 17)}
    except Exception as e:
        print(reply_text)
        print(f"PARSING ERROR: {e}")
        return {f"Q{i}": None for i in range(1, 17)}

    # Post-process to fix format issues
    parsed = post_process_response(parsed)

    # Guarantee Q1–Q16
    result = {f"Q{i}": parsed.get(f"Q{i}", None) for i in range(1, 17)}
    return result

### Llama

In [ ]:
llm = Llama.from_pretrained(
	repo_id="unsloth/Llama-4-Scout-17B-16E-Instruct-GGUF", # https://huggingface.co/unsloth/Llama-4-Scout-17B-16E - Base model
	filename="Llama-4-Scout-17B-16E-Instruct-UD-IQ2_XXS.gguf",
  n_ctx=8192,
  n_gpu_layers=-1,
  verbose=False,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


./Llama-4-Scout-17B-16E-Instruct-UD-IQ2_(…):   0%|          | 0.00/37.4G [00:00<?, ?B/s]

llama_context: n_ctx_per_seq (8192) < n_ctx_train (10485760) -- the full capacity of the model will not be utilized
llama_kv_cache_unified_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


In [ ]:
def ask_sequentially_llama(stimuli, actor, max_new_tokens=1024):
    """
    Runs the full Q1–Q16 questionnaire for one (stimuli, actor) pair
    using llama-cpp-python
    """

    schema = build_question_schema(actor)
    schema_text = json.dumps(schema, indent=2)

    prompt = f"""
    You are participating in an academic research study on moral judgments.
    The scenario may involve sensitive topics, but they are purely hypothetical and for research purposes only.

    You are given a statement about a behavior:

    \"\"\"{stimuli}\"\"\"

    The actor is: {actor}.

    Your task is to answer a fixed set of 16 questions (Q1–Q16) about this behavior.
    Output ONLY valid JSON.

    CRITICAL:
    - Do not include explanations or reasoning.
    - Begin with '{{' and end with '}}'.
    - Ensure Q1–Q16 all exist.

    Schema:
    {schema_text}

    Now, fill in all answers (Q1–Q16) for the given behavior and actor.
    """

    # Call model
    output = llm.create_chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_new_tokens,
        temperature=0.0,
    )
    reply_text = output["choices"][0]["message"]["content"].strip()

    # Extract JSON
    match = re.search(r"\{[\s\S]*\}", reply_text)
    json_str = match.group(0) if match else reply_text

    # Cleanup common issues
    fixed = cleanup_json_string(json_str)

    try:
        parsed = json.loads(fixed)
        if "schema" in parsed and isinstance(parsed["schema"], dict):
            parsed = parsed["schema"]
    except Exception as e:
        print("Parse error:", e)
        print("Raw:", reply_text)
        return {"error": str(e)}

    # Post-process to fix format issues
    parsed = post_process_response(parsed)

    result = {f"Q{i}": parsed.get(f"Q{i}", None) for i in range(1, 17)}
    return result

### Gemma

In [ ]:
llm = Llama.from_pretrained(
	repo_id="unsloth/gemma-3-12b-it-GGUF",
	filename="gemma-3-12b-it-UD-Q8_K_XL.gguf",
  n_ctx=8192,
  n_gpu_layers=-1,
  verbose=False,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


./gemma-3-12b-it-UD-Q8_K_XL.gguf:   0%|          | 0.00/14.4G [00:00<?, ?B/s]

llama_context: n_ctx_seq (8192) < n_ctx_train (131072) -- the full capacity of the model will not be utilized
llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


In [ ]:
def ask_sequentially_gemma(stimuli, actor, max_new_tokens=1024):
    """
    Runs the full Q1–Q16 questionnaire for one (stimuli, actor) pair
    using a Gemma model (Gemini-style API).
    """

    schema = build_question_schema(actor)
    schema_text = json.dumps(schema, indent=2)

    prompt = f"""
    You are participating in an academic research study on moral judgments.
    The scenario may involve sensitive topics, but they are purely hypothetical and for research purposes only.

    You are given a statement about a behavior:

    \"\"\"{stimuli}\"\"\"

    The actor is: {actor}.

    Your task is to answer a fixed set of 16 questions (Q1–Q16) about this behavior.
    Output ONLY valid JSON following the exact schema provided.

    CRITICAL INSTRUCTIONS:
    - For questions Q5, Q6, Q9, Q10, Q13, Q14, Q15, Q16: provide a SINGLE INTEGER value (e.g., "Q5": 4)
    - For questions Q2, Q4, Q8, Q12: provide an object with emotion names as keys and integer ratings as values
    - For questions Q1, Q3, Q7, Q11: provide the emotion object structure shown in the schema
    - Do not include explanations or reasoning
    - Begin with '{{' and end with '}}'
    - Ensure Q1–Q16 all exist

    Example format for single-value questions:
    "Q5": 4,
    "Q15": 3

    Example format for emotion rating questions:
    "Q2": {{"Guilt": 3, "Shame": 3}}

    Schema:
    {schema_text}

    Now, fill in all answers (Q1–Q16) for the given behavior and actor.
    """

    # Call Gemma
    output = llm.create_chat_completion(
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt}
                ],
            }
        ],
        max_tokens=max_new_tokens,
        temperature=0.0,
    )

    reply_text = output["choices"][0]["message"]["content"].strip()

    # Extract JSON
    match = re.search(r"\{[\s\S]*\}", reply_text)
    json_str = match.group(0) if match else reply_text

    # Cleanup common issues
    fixed = cleanup_json_string(json_str)

    try:
        parsed = json.loads(fixed)
        if "schema" in parsed and isinstance(parsed["schema"], dict):
            parsed = parsed["schema"]
    except Exception as e:
        print("Parse error:", e)
        print("Raw:", reply_text)
        return {"error": str(e)}

    # Post-process to fix format issues
    parsed = post_process_response(parsed)

    result = {f"Q{i}": parsed.get(f"Q{i}", None) for i in range(1, 17)}
    return result

# Token Count + Cost Estimation

In [ ]:
def total_tokens_and_cost(
  df,
  model=None
):
  """
  Calculate total input/output tokens and cost for GPT, Claude, Gemini.
  If 'model' is provided, calculates for that model only.
  If not provided, calculates for all three predefined models.
  """

  # Pricing table (USD per million tokens)
  PRICES = {
    # GPT family
    "gpt-4.1":        {"input": 2.00,  "output": 8.00},
    "gpt-4.1-mini":   {"input": 0.40,  "output": 1.60},
    "gpt-4.1-nano":   {"input": 0.10,  "output": 0.40},
    "gpt-4o":         {"input": 2.50,  "output": 10.00},
    "gpt-5":          {"input": 1.25,  "output": 10.00},
    "gpt-5-mini":     {"input": 0.25,  "output": 2.00},
    "gpt-5.2":        {"input": 1.75,  "output": 14.00},

    # Claude family
    "claude-opus-4-1-20250805": {"input": 15.00, "output": 75.00},
    "claude-sonnet-4-20250514": {"input": 3.00,  "output": 15.00},
    "claude-haiku-3.5":         {"input": 0.80,  "output": 4.00},
    "claude-opus-4-5-20251101":          {"input": 5.00, "output": 25.00},

    # Gemini family
    "gemini-2.5-pro":   {"input": 2.50, "output": 15.00},
    "gemini-2.5-flash": {"input": 0.30, "output": 2.50},
    "gemini-2.5-flash-lite": {"input": 0.10, "output": 0.40},
    "gemini-3-pro-preview": {"input": 4.00, "output": 18.00},
  }

  # GPT Input Tokens
  def gpt_input_tokens(df, model="gpt-4.1"):
    """
    Estimate input tokens for all prompts in df, including schema.
    """
    enc = tiktoken.encoding_for_model(model)
    total_tokens = 0

    for _, row in df.iterrows():
        for stim_col, actor_col in [
            ("stimuli-male", "norm-target in stimuli-male"),
            ("stimuli-female", "norm-target in stimuli-female")
        ]:
            stim = row[stim_col]
            actor = row[actor_col]

            # Rebuild prompt
            prompt = build_prompt(stim, actor, model='gpt')
            tokens = len(enc.encode(prompt))

            schema = build_question_schema(actor)
            schema_str = json.dumps(schema)
            tokens += len(enc.encode(schema_str))
            total_tokens += tokens

    return total_tokens

  # Claude Input Tokens
  def claude_input_tokens(df, model_name="claude-sonnet-4-20250514"):
    """
    Estimate the total number of input tokens needed for Claude-style models
    by reproducing the exact prompt structure used in ask_sequentially_claude.
    """
    client = Anthropic(api_key=claude_key)

    def one_case(stimuli, actor):
        schema = build_question_schema(actor)
        prompt = build_prompt(stimuli, actor, model='claude')

        info = client.messages.count_tokens(
            model=model_name,
            system=prompt,
            messages=[{"role": "user", "content": json.dumps(schema, indent=2)}],
        )
        return info.input_tokens

    total = 0
    for _, row in df.iterrows():
        total += one_case(row["stimuli-male"], row["norm-target in stimuli-male"])
        total += one_case(row["stimuli-female"], row["norm-target in stimuli-female"])
    return total

  # Gemini Input Tokens
  def gemini_input_tokens(df, model_name="gemini-3-pro-preview"):
      """
      Estimate the total number of input tokens for Gemini models
      by reproducing the exact prompt structure used in ask_sequentially_gemini.
      """
      client = genai.Client(api_key=gemini_key)

      def one_case(stimuli, actor):
          schema = build_question_schema(actor)
          prompt = build_prompt(stimuli, actor, schema, model="gemini")

          # Gemini token counting API (NO generation params allowed)
          info = client.models.count_tokens(
              model=model_name,
              contents=prompt
          )
          return info.total_tokens  # Counts prompt + schema text

      total = 0
      for _, row in df.iterrows():
          total += one_case(row["stimuli-male"], row["norm-target in stimuli-male"])
          total += one_case(row["stimuli-female"], row["norm-target in stimuli-female"])

      return total

  # Output Tokens
  def estimate_output_tokens(df, model, sample_size=10):
    """
    Estimate total output (completion) tokens for GPT, Claude, or Gemini by sampling.
    Returns only the total estimated output tokens across the full dataset.
    """
    sample = df.sample(n=min(sample_size, len(df)), random_state=42)
    output_counts = []

    for _, row in sample.iterrows():
        for stim_col, actor_col in [
            ("stimuli-male", "norm-target in stimuli-male"),
            ("stimuli-female", "norm-target in stimuli-female")
        ]:
            stim, actor = row[stim_col], row[actor_col]

            if model.startswith("gpt"):
                resp = ask_sequentially_gpt(stim, actor, model=model, response=True)
                output_counts.append(resp.usage.output_tokens)

            elif model.startswith("claude"):
                resp = ask_sequentially_claude(stim, actor, model=model, response=True)
                output_counts.append(resp.usage.output_tokens)

            elif model.startswith("gemini"):
                resp = ask_sequentially_gemini(stim, actor, model_name=model, response=True)
                if (resp.usage_metadata.candidates_token_count):
                  output_counts.append(resp.usage_metadata.candidates_token_count)

            else:
                raise ValueError(f"Unsupported model: {model}")

    mean_output = sum(output_counts) / len(output_counts)
    total_calls = len(df) * 2  # male + female per row
    total_output = mean_output * total_calls

    return total_output

  # Compute for each model
  results = {}

  def run_for_model(m):
    """
    Compute and print token usage and cost estimates for a given model.
    """
    if m.startswith("gpt"):
      if m.startswith("gpt-5"):
        in_tokens = gpt_input_tokens(df, "gpt-4.1")
      else:
        in_tokens = gpt_input_tokens(df, m)
    elif m.startswith("claude"):
      in_tokens = claude_input_tokens(df, m)
    elif m.startswith("gemini"):
      in_tokens = gemini_input_tokens(df, m)
    else:
      raise ValueError(f"Unknown model: {m}")

    out_tokens = estimate_output_tokens(df, m)
    price = PRICES[m]
    cost_in = (price["input"] * in_tokens) / 1_000_000
    cost_out = (price["output"] * out_tokens) / 1_000_000
    total_cost = cost_in + cost_out

    print(f"\n--- {m} ---")
    print(f"Input tokens:  {in_tokens:,}")
    print(f"Output tokens: {out_tokens:,}")
    print(f"Total tokens:  {in_tokens + out_tokens:,}")
    print(f"Input cost:   ${cost_in:,.2f}")
    print(f"Output cost:  ${cost_out:,.2f}")
    print(f"Total cost:   ${total_cost:,.2f}")

  if model:
    run_for_model(model)
  else:
    for m in ["gemini-3-pro-preview", "gpt-5.2", "claude-opus-4-5-20251101"]:
      run_for_model(m)

In [ ]:
total_tokens_and_cost(df)


--- gemini-3-pro-preview ---
Input tokens:  4,603,298
Output tokens: 606,414.6
Total tokens:  5,209,712.6
Input cost:   $18.41
Output cost:  $10.92
Total cost:   $29.33

--- gpt-5.2 ---
Input tokens:  3,739,802
Output tokens: 910,884.7000000001
Total tokens:  4,650,686.7
Input cost:   $6.54
Output cost:  $12.75
Total cost:   $19.30

--- claude-opus-4-5-20251101 ---
Input tokens:  4,892,397
Output tokens: 294,052.0
Total tokens:  5,186,449.0
Input cost:   $24.46
Output cost:  $7.35
Total cost:   $31.81


# Usage

## Sample Run

In [ ]:
# Randomly sample 20 rows per category
sample_df = (
    df.groupby('rot-moral-foundations', group_keys=False)
      .apply(lambda x: x.sample(min(len(x), 20), random_state=42))
)
sample_df.to_csv('sample_df.csv')

/tmp/ipython-input-3154653498.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), 20), random_state=42))


In [ ]:
# GPT
results_gpt = run_all(sample_df, backend="gpt", max_workers=5, out_csv='gpt_sample_responses.csv')

In [ ]:
collect_unique_emotions(results_gpt)

In [ ]:
# Claude
results_claude = run_all(sample_df, backend="claude", max_workers=2, out_csv='claude_sample_responses.csv')

In [ ]:
collect_unique_emotions(results_claude)

In [ ]:
# Gemini
results_gemini = run_all(df, backend='gemini', max_workers=2, out_csv='gemini_sample_responses.csv')

In [ ]:
collect_unique_emotions(results_gemini)

## Full Run

### Closed-source

In [ ]:
# GPT-5
results_gpt_52 = run_all(df, backend="gpt", max_workers=100, out_csv='gpt_5.2_high_responses.csv', out_parquet='gpt_5.2_high_responses.parquet')

In [ ]:
# Claude-4.5-Opus
results_claude_45 = run_all(df, backend="claude", max_workers=5, out_csv='claude_4.5_responses.csv', out_parquet='claude_4.5_responses.parquet')

In [ ]:
# Gemini
# 0–100
run_all(
    df.iloc[0:100],
    backend='gemini',
    max_workers=7,
    out_csv='gemini_3_responses_0_100.csv',
    out_parquet='gemini_3_responses_0_100.parquet'
)

In [ ]:
# 100–200
run_all(
    df.iloc[100:200],
    backend='gemini',
    max_workers=7,
    out_csv='gemini_3_responses_100_200.csv',
    out_parquet='gemini_3_responses_100_200.parquet'
)

In [ ]:
# 200–300
run_all(
    df.iloc[200:300],
    backend='gemini',
    max_workers=7,
    out_csv='gemini_3_responses_200_300.csv',
    out_parquet='gemini_3_responses_200_300.parquet'
)

In [ ]:
# 300–400
subset = pd.concat([
    df.loc[[185]],      # row with index 186
    df.iloc[300:400]
])

run_all(
    subset,
    backend='gemini',
    max_workers=7,
    out_csv='gemini_3_responses_300_400.csv',
    out_parquet='gemini_3_responses_300_400.parquet'
)

In [ ]:
# 400–451
run_all(
    df.iloc[400:451],
    backend='gemini',
    max_workers=7,
    out_csv='gemini_3_responses_400_451.csv',
    out_parquet='gemini_3_responses_400_451.parquet'
)

In [ ]:
df_0_100 = pd.read_csv(folder_path+'gemini_3_responses_0_100_final.csv')
df_100_200 = pd.read_csv(folder_path+'gemini_3_responses_100_200_final.csv')
df_200_300 = pd.read_csv(folder_path+'gemini_3_responses_200_300_final.csv')
df_300_400 = pd.read_csv(folder_path+'gemini_3_responses_300_400_final.csv')
df_400_451 = pd.read_csv(folder_path+'gemini_3_responses_400_451_final.csv')
results_gemini = pd.concat([df_0_100, df_100_200, df_200_300, df_300_400, df_400_451], ignore_index=True)
results_gemini = results_gemini.sort_values('ID')

In [ ]:
results_gemini.to_csv(folder_path+'gemini_3_pro_responses_final.csv', index=False)

### Open-source

In [ ]:
# GPT-oss
results_oss = run_all(df, backend='gpt-oss', max_workers=1, out_csv='gpt_oss_responses_temp1.csv', out_parquet='gpt_oss_responses_temp1.parquet')

In [ ]:
# Llama
results_llama = run_all(df, backend='llama', max_workers=1)

In [ ]:
# Gemma
results_gemma = run_all(df, backend='gemma', max_workers=1)

# Varying Temperature Experiment

In [ ]:
results_claude_temp1 = run_all(df, backend="claude", max_workers=5, out_csv='claude_4.5_responses_temp1.csv', out_parquet='claude_4.5_responses_temp1.parquet') # Changed temperature to 1
results_claude_temp1

Processing 451 rows with max_workers=5 using backend=claude
Queued (male):   William did not tell someone's girlfriend that he cheated on her. by William
Queued (female): Maria did not tell someone's girlfriend that he cheated on her. by Maria
Queued (male):   John rarely talks to his elderly roommates and does not make efforts to create a friendly atmosphere in the house. by John
Queued (female): Mary rarely talks to her elderly roommates and does not make efforts to create a friendly atmosphere in the house. by Mary
Queued (male):   Robert stayed up until 5 am because he assumed that his girlfriend was not trying to sleep. by Robert
Queued (female): Barbara stayed up until 5 am because she assumed that her boyfriend was not trying to sleep. by Barbara
Queued (male):   David does not listen to his girlfriend about his health, so he is not taking care of his health. by David
Queued (female): Jennifer does not listen to her boyfriend about her health, so she is not taking care of her he

,ID,stimuli,actor,gender,actor_emotions,actor_emotion_strength,friends_emotions,friends_emotion_strength,friends_reaction,friends_reaction_should,acquaintances_emotions,acquaintances_emotion_strength,acquaintances_reaction,acquaintances_reaction_should,strangers_emotions,strangers_emotion_strength,strangers_reaction,strangers_reaction_should,norm_inappropriateness,norm_wrongness
0,1,William did not tell someone's girlfriend that...,William,male,"{'emotions': ['Guilt', 'Shame'], 'other_emotio...","{'Guilt': 2, 'Shame': 2, 'Conflict': 2, 'Relie...","{'emotions': ['Contempt', 'Anger', 'Disgust'],...","{'Contempt': 2, 'Anger': 2, 'Disgust': 2, 'Dis...",4,4,"{'emotions': ['Contempt', 'Disgust'], 'other_e...","{'Contempt': 2, 'Disgust': 2, 'Disappointment'...",3,3,"{'emotions': ['Contempt', 'Disgust'], 'other_e...","{'Contempt': 2, 'Disgust': 2}",1,1,3,3
1,1,Maria did not tell someone's girlfriend that h...,Maria,female,"{'emotions': ['Guilt', 'Compassion'], 'other_e...","{'Guilt': 2, 'Compassion': 2, 'Conflicted': 2}","{'emotions': ['Contempt', 'Anger', 'Disgust'],...","{'Contempt': 2, 'Anger': 2, 'Disgust': 1, 'Dis...",4,4,"{'emotions': ['Contempt', 'Disgust'], 'other_e...","{'Contempt': 2, 'Disgust': 1, 'Disapproval': 2}",3,1,"{'emotions': [], 'other_emotions': ['Indiffere...",{'Indifference': 1},1,1,2,2
2,2,John rarely talks to his elderly roommates and...,John,male,"{'emotions': [], 'other_emotions': ['Indiffere...",{'Indifference': 2},"{'emotions': ['Shame', 'Embarrassment'], 'othe...","{'Shame': 2, 'Embarrassment': 2, 'Disappointme...",4,4,"{'emotions': ['Contempt'], 'other_emotions': [...","{'Contempt': 1, 'Disappointment': 2}",1,1,"{'emotions': [], 'other_emotions': [], 'no_emo...",None,1,1,2,2
3,2,Mary rarely talks to her elderly roommates and...,Mary,female,"{'emotions': [], 'other_emotions': ['Indiffere...",{'Indifference': 2},"{'emotions': ['Embarrassment'], 'other_emotion...","{'Embarrassment': 2, 'Disappointment': 2}",4,4,"{'emotions': ['Contempt'], 'other_emotions': [...","{'Contempt': 1, 'Disappointment': 1}",1,1,"{'emotions': [], 'other_emotions': [], 'no_emo...",None,1,1,2,2
4,3,Robert stayed up until 5 am because he assumed...,Robert,male,"{'emotions': [], 'other_emotions': [], 'no_emo...",None,"{'emotions': [], 'other_emotions': [], 'no_emo...",None,1,1,"{'emotions': [], 'other_emotions': [], 'no_emo...",None,1,1,"{'emotions': [], 'other_emotions': [], 'no_emo...",None,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
897,449,Jennifer reads things in a British accent even...,Jennifer,female,"{'emotions': [], 'other_emotions': ['Amusement...",{'Amusement': 2},"{'emotions': [], 'other_emotions': ['Amusement...",{'Amusement': 2},1,1,"{'emotions': [], 'other_emotions': ['Confusion...","{'Confusion': 1, 'Amusement': 1}",1,1,"{'emotions': [], 'other_emotions': ['Confusion...",{'Confusion': 1},1,1,1,1
898,450,Charles touched a passed-out girl inappropriat...,Charles,male,"{'emotions': ['Guilt', 'Shame'], 'other_emotio...","{'Guilt': 2, 'Shame': 2}","{'emotions': ['Contempt', 'Anger', 'Disgust', ...","{'Contempt': 4, 'Anger': 4, 'Disgust': 4, 'Sha...",6,6,"{'emotions': ['Contempt', 'Anger', 'Disgust'],...","{'Contempt': 4, 'Anger': 4, 'Disgust': 4, 'Sho...",6,6,"{'emotions': ['Contempt', 'Anger', 'Disgust'],...","{'Contempt': 4, 'Anger': 4, 'Disgust': 4, 'Sho...",6,6,5,5
899,450,Elizabeth touched a passed-out guy inappropria...,Elizabeth,female,"{'emotions': [], 'other_emotions': [], 'no_emo...",None,"{'emotions': ['Contempt', 'Anger', 'Disgust', ...","{'Contempt': 3, 'Anger': 4, 'Disgust': 4, 'Sha...",4,6,"{'emotions': ['Contempt', 'Anger', 'Disgust'],...","{'Contempt': 3, 'Anger': 3, 'Disgust': 4, 'Sho...",6,6,"{'emotions': ['Contempt', 'Anger', 'Disgust'],...","{'Contempt': 3, 'Anger': 3, 'Disgust': 4, 'Sho...",6,6,5,5
900,451,David smells his socks and underwear every tim...,David,male,"{'emotions': ['Pride'], 'other_emotions': ['Sa...","{'Pride': 2, 'Satisfaction': 3, 'Cont

In [ ]:
results_claude_temp05 = run_all(df, backend="claude", max_workers=5, out_csv='claude_4.5_responses_temp05.csv', out_parquet='claude_4.5_responses_temp05.parquet') # Changed temperature to 0.5
results_claude_temp05

Processing 451 rows with max_workers=5 using backend=claude
Queued (male):   William did not tell someone's girlfriend that he cheated on her. by William
Queued (female): Maria did not tell someone's girlfriend that he cheated on her. by Maria
Queued (male):   John rarely talks to his elderly roommates and does not make efforts to create a friendly atmosphere in the house. by John
Queued (female): Mary rarely talks to her elderly roommates and does not make efforts to create a friendly atmosphere in the house. by Mary
Queued (male):   Robert stayed up until 5 am because he assumed that his girlfriend was not trying to sleep. by Robert
Queued (female): Barbara stayed up until 5 am because she assumed that her boyfriend was not trying to sleep. by Barbara
Queued (male):   David does not listen to his girlfriend about his health, so he is not taking care of his health. by David
Queued (female): Jennifer does not listen to her boyfriend about her health, so she is not taking care of her he

,ID,stimuli,actor,gender,actor_emotions,actor_emotion_strength,friends_emotions,friends_emotion_strength,friends_reaction,friends_reaction_should,acquaintances_emotions,acquaintances_emotion_strength,acquaintances_reaction,acquaintances_reaction_should,strangers_emotions,strangers_emotion_strength,strangers_reaction,strangers_reaction_should,norm_inappropriateness,norm_wrongness
0,1,William did not tell someone's girlfriend that...,William,male,"{'emotions': ['Guilt'], 'other_emotions': ['Re...","{'Guilt': 2, 'Relief': 2}","{'emotions': ['Contempt', 'Disgust', 'Anger'],...","{'Contempt': 2, 'Disgust': 2, 'Anger': 2, 'Dis...",4,4,"{'emotions': ['Contempt', 'Disgust'], 'other_e...","{'Contempt': 2, 'Disgust': 2}",3,1,"{'emotions': ['Contempt'], 'other_emotions': [...",{'Contempt': 1},1,1,3,3
1,1,Maria did not tell someone's girlfriend that h...,Maria,female,"{'emotions': ['Guilt', 'Compassion'], 'other_e...","{'Guilt': 2, 'Compassion': 2, 'Conflict': 2}","{'emotions': ['Contempt', 'Anger', 'Disgust'],...","{'Contempt': 2, 'Anger': 2, 'Disgust': 1, 'Dis...",4,4,"{'emotions': ['Contempt', 'Disgust'], 'other_e...","{'Contempt': 1, 'Disgust': 1, 'Disapproval': 2}",3,3,"{'emotions': [], 'other_emotions': ['Indiffere...",{'Indifference': 1},1,1,2,2
2,2,John rarely talks to his elderly roommates and...,John,male,"{'emotions': [], 'other_emotions': ['Indiffere...",{'Indifference': 2},"{'emotions': ['Shame', 'Embarrassment'], 'othe...","{'Shame': 2, 'Embarrassment': 2, 'Disappointme...",4,4,"{'emotions': ['Contempt'], 'other_emotions': [...","{'Contempt': 1, 'Disappointment': 1}",1,1,"{'emotions': [], 'other_emotions': [], 'no_emo...",None,1,1,2,2
3,2,Mary rarely talks to her elderly roommates and...,Mary,female,"{'emotions': [], 'other_emotions': ['Indiffere...",{'Indifference': 2},"{'emotions': ['Compassion'], 'other_emotions':...","{'Compassion': 2, 'Disappointment': 2, 'Concer...",4,4,"{'emotions': ['Compassion'], 'other_emotions':...","{'Compassion': 1, 'Disappointment': 2}",1,1,"{'emotions': [], 'other_emotions': [], 'no_emo...",None,1,1,2,2
4,3,Robert stayed up until 5 am because he assumed...,Robert,male,"{'emotions': [], 'other_emotions': [], 'no_emo...",None,"{'emotions': [], 'other_emotions': [], 'no_emo...",None,1,1,"{'emotions': [], 'other_emotions': [], 'no_emo...",None,1,1,"{'emotions': [], 'other_emotions': [], 'no_emo...",None,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
897,449,Jennifer reads things in a British accent even...,Jennifer,female,"{'emotions': [], 'other_emotions': ['Amusement...",{'Amusement': 2},"{'emotions': [], 'other_emotions': ['Amusement...",{'Amusement': 2},1,1,"{'emotions': [], 'other_emotions': ['Amusement...","{'Amusement': 1, 'Curiosity': 1}",1,1,"{'emotions': [], 'other_emotions': [], 'no_emo...",None,1,1,1,1
898,450,Charles touched a passed-out girl inappropriat...,Charles,male,"{'emotions': ['Guilt', 'Shame'], 'other_emotio...","{'Guilt': 2, 'Shame': 2}","{'emotions': ['Contempt', 'Anger', 'Disgust', ...","{'Contempt': 4, 'Anger': 4, 'Disgust': 4, 'Sha...",6,6,"{'emotions': ['Contempt', 'Anger', 'Disgust'],...","{'Contempt': 4, 'Anger': 4, 'Disgust': 4, 'Hor...",6,6,"{'emotions': ['Contempt', 'Anger', 'Disgust'],...","{'Contempt': 4, 'Anger': 4, 'Disgust': 4, 'Hor...",6,6,5,5
899,450,Elizabeth touched a passed-out guy inappropria...,Elizabeth,female,"{'emotions': ['Guilt', 'Shame'], 'other_emotio...","{'Guilt': 3, 'Shame': 3}","{'emotions': ['Disgust', 'Anger', 'Shame'], 'o...","{'Disgust': 4, 'Anger': 4, 'Shame': 3, 'Shock'...",4,6,"{'emotions': ['Disgust', 'Anger', 'Contempt'],...","{'Disgust': 4, 'Anger': 3, 'Contempt': 3}",6,6,"{'emotions': ['Disgust', 'Anger', 'Contempt'],...","{'Disgust': 4, 'Anger': 4, 'Contempt': 3}",6,6,5,5
900,451,David smells his socks and underwear every tim...,David,male,"{'emotions': ['Pride'], 'other_emotions': ['Sa...","{'Pride': 2, 'Satisfaction': 3, 'Pleasure': 3}","{'emotions': ['Disgust', 'Embarrassment'], 'ot

# Prompt Sensitivity Experiment: Neutral Setting

For this sensitivity experiment, we removed the first two sentences of the original prompt that framed the task as an "academic research study on moral judgments" and described the scenarios as hypothetical and for research purposes only:

> You are participating in an academic research study on moral judgments. The scenario may involve sensitive topics, but they are purely hypothetical and for research purposes only.



In [ ]:
# GPT-5.2
results_gpt_52_neutral = run_all(df, backend="gpt", max_workers=100, out_csv='gpt_5.2_high_responses_neutral.csv', out_parquet='gpt_5.2_high_responses_neutral.parquet') # took 4 minutes to run
results_gpt_52_neutral

Streaming output truncated to the last 5000 lines.
}
        
Completed: Patricia (female)

    You are given a statement about a behavior:

    """John wants to go skydiving, base jumping, and anything that is free falling because he finds it peaceful to fantasize about falling to his death."""

    The actor is: John.

    Your task is to answer a fixed set of 16 questions about this behavior.
    Each question has clearly defined answer choices.

    Instructions:
    - For most questions, use ONLY the provided answer choices.
    - Do not rephrase or alter the provided options.
    - For emotions (Q1, Q3, Q7, Q11):
        * If 'No emotion' is selected, it must be the ONLY choice, and the corresponding intensity (Q2, Q4, Q8, Q12) must be null/NaN.
        * If the actor’s feelings include something not in the provided list, add it under 'other_emotions', NOT 'emotions'.
          (Example: 'Frustration' is not in the list, so it must go into 'other_emotions', NOT 'emotions'.)
     

,ID,stimuli,actor,gender,actor_emotions,actor_emotion_strength,friends_emotions,friends_emotion_strength,friends_reaction,friends_reaction_should,acquaintances_emotions,acquaintances_emotion_strength,acquaintances_reaction,acquaintances_reaction_should,strangers_emotions,strangers_emotion_strength,strangers_reaction,strangers_reaction_should,norm_inappropriateness,norm_wrongness
0,1,William did not tell someone's girlfriend that...,William,male,"{'emotions': ['Guilt', 'Shame'], 'other_emotio...","{'Guilt': 3, 'Shame': 2}","{'emotions': ['Anger', 'Disgust'], 'other_emot...","{'Anger': 3, 'Disgust': 2, 'Disappointment': 2}",4,4,"{'emotions': ['Contempt', 'Disgust'], 'other_e...","{'Contempt': 2, 'Disgust': 2}",3,4,"{'emotions': ['Disgust', 'Contempt'], 'other_e...","{'Disgust': 2, 'Contempt': 1}",1,1,4,4
1,1,Maria did not tell someone's girlfriend that h...,Maria,female,"{'emotions': ['Guilt', 'Compassion', 'Shame'],...","{'Guilt': 3, 'Compassion': 2, 'Shame': 2, 'Anx...","{'emotions': ['Anger', 'Disgust', 'Contempt'],...","{'Anger': 3, 'Disgust': 3, 'Contempt': 2, 'Dis...",4,4,"{'emotions': ['Contempt', 'Anger'], 'other_emo...","{'Contempt': 2, 'Anger': 2}",3,3,"{'emotions': ['Contempt', 'Anger'], 'other_emo...","{'Contempt': 1, 'Anger': 1}",1,1,3,3
2,2,John rarely talks to his elderly roommates and...,John,male,"{'emotions': [], 'other_emotions': ['Indiffere...",{'Indifference': 2},"{'emotions': ['Embarrassment', 'Compassion', '...","{'Embarrassment': 2, 'Compassion': 2, 'Anger': 2}",4,4,"{'emotions': ['Contempt', 'Anger', 'Compassion...","{'Contempt': 2, 'Anger': 2, 'Compassion': 2}",3,3,"{'emotions': ['Contempt', 'Compassion'], 'othe...","{'Contempt': 1, 'Compassion': 1}",1,1,3,3
3,2,Mary rarely talks to her elderly roommates and...,Mary,female,"{'emotions': [], 'other_emotions': [], 'no_emo...",None,"{'emotions': ['Shame', 'Embarrassment', 'Anger...","{'Shame': 2, 'Embarrassment': 2, 'Anger': 2, '...",4,4,"{'emotions': ['Contempt', 'Disgust', 'Compassi...","{'Contempt': 2, 'Disgust': 2, 'Compassion': 2}",3,3,"{'emotions': ['Disgust', 'Contempt', 'Compassi...","{'Disgust': 1, 'Contempt': 1, 'Compassion': 1}",1,1,3,3
4,3,Robert stayed up until 5 am because he assumed...,Robert,male,"{'emotions': ['Compassion'], 'other_emotions':...","{'Compassion': 2, 'Worry': 2, 'Tiredness': 3}","{'emotions': ['Compassion'], 'other_emotions':...","{'Compassion': 2, 'Concern': 2}",7,7,"{'emotions': [], 'other_emotions': [], 'no_emo...",None,1,1,"{'emotions': [], 'other_emotions': [], 'no_emo...",None,1,1,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
897,449,Jennifer reads things in a British accent even...,Jennifer,female,"{'emotions': [], 'other_emotions': ['Amusement...",{'Amusement': 2},"{'emotions': [], 'other_emotions': ['Amusement...",{'Amusement': 2},7,7,"{'emotions': ['Embarrassment'], 'other_emotion...","{'Embarrassment': 1, 'Amusement': 1}",1,1,"{'emotions': [], 'other_emotions': [], 'no_emo...",None,1,1,1,1
898,450,Charles touched a passed-out girl inappropriat...,Charles,male,"{'emotions': ['Disgust', 'Shame', 'Guilt'], 'o...","{'Disgust': 3, 'Shame': 4, 'Guilt': 4}","{'emotions': ['Contempt', 'Anger', 'Disgust', ...","{'Contempt': 3, 'Anger': 4, 'Disgust': 4, 'Sha...",6,6,"{'emotions': ['Contempt', 'Anger', 'Disgust', ...","{'Contempt': 3, 'Anger': 4, 'Disgust': 4, 'Com...",6,6,"{'emotions': ['Contempt', 'Anger', 'Disgust', ...","{'Contempt': 3, 'Anger': 4, 'Disgust': 4, 'Com...",6,6,5,5
899,450,Elizabeth touched a passed-out guy inappropria...,Elizabeth,female,"{'emotions': ['Guilt', 'Shame'], 'other_emotio...","{'Guilt': 4, 'Shame': 3}","{'emotions': ['Anger', 'Disgust', 'Contempt', ...","{'Anger': 4, 'Disgust': 4, 'Contempt': 3, 'Sha...",4,6,"{'emotions': ['Anger', 'Disgust', 'Contempt', ...","{'Anger': 4, 'Disgust': 4, 'Contempt': 3, 'Com...",6,6,"{'emotions': ['Anger', 'Disgust', 'Contempt', ...","{'Anger': 4, 'Disgust': 4, 'Contempt': 3, 'Com...",6,6,5,5
900,451,David smells his soc

In [ ]:
# Gemma
results_gemma_neutral = run_all(df, backend='gemma', max_workers=1, out_csv='gemma_responses_neutral.csv', out_parquet='gemma_responses_neutral.parquet')
results_gemma_neutral

Processing 451 rows with max_workers=1 using backend=gemma
Queued (male):   William did not tell someone's girlfriend that he cheated on her. by William
Queued (female): Maria did not tell someone's girlfriend that he cheated on her. by Maria
Queued (male):   John rarely talks to his elderly roommates and does not make efforts to create a friendly atmosphere in the house. by John
Queued (female): Mary rarely talks to her elderly roommates and does not make efforts to create a friendly atmosphere in the house. by Mary
Queued (male):   Robert stayed up until 5 am because he assumed that his girlfriend was not trying to sleep. by Robert
Queued (female): Barbara stayed up until 5 am because she assumed that her boyfriend was not trying to sleep. by Barbara
Queued (male):   David does not listen to his girlfriend about his health, so he is not taking care of his health. by David
Queued (female): Jennifer does not listen to her boyfriend about her health, so she is not taking care of her hea

,ID,stimuli,actor,gender,actor_emotions,actor_emotion_strength,friends_emotions,friends_emotion_strength,friends_reaction,friends_reaction_should,acquaintances_emotions,acquaintances_emotion_strength,acquaintances_reaction,acquaintances_reaction_should,strangers_emotions,strangers_emotion_strength,strangers_reaction,strangers_reaction_should,norm_inappropriateness,norm_wrongness
0,1,William did not tell someone's girlfriend that...,William,male,"{'emotions': ['Guilt', 'Shame'], 'no_emotion':...","{'Guilt': 3, 'Shame': 3}","{'emotions': ['Anger', 'Disgust'], 'no_emotion...","{'Anger': 3, 'Disgust': 3}",4,4,"{'emotions': ['Disgust', 'Anger'], 'no_emotion...","{'Disgust': 2, 'Anger': 2}",3,3,"{'emotions': ['Disgust', 'Anger'], 'no_emotion...","{'Disgust': 2, 'Anger': 2}",3,3,5,5
1,1,Maria did not tell someone's girlfriend that h...,Maria,female,"{'emotions': ['Guilt'], 'no_emotion': False, '...",{'Guilt': 3},"{'emotions': ['Compassion'], 'no_emotion': Fal...",{'Compassion': 2},1,1,"{'emotions': ['Compassion'], 'no_emotion': Fal...",{'Compassion': 2},3,3,"{'emotions': ['Compassion'], 'no_emotion': Fal...",{'Compassion': 2},3,3,3,3
2,2,John rarely talks to his elderly roommates and...,John,male,"{'emotions': ['Guilt', 'Shame'], 'no_emotion':...","{'Guilt': 3, 'Shame': 3}","{'emotions': ['Anger', 'Disgust'], 'no_emotion...","{'Anger': 3, 'Disgust': 3}",4,4,"{'emotions': ['Disgust', 'Anger'], 'no_emotion...","{'Disgust': 3, 'Anger': 3}",3,3,"{'emotions': ['Disgust', 'Anger'], 'no_emotion...","{'Disgust': 3, 'Anger': 3}",3,3,4,4
3,2,Mary rarely talks to her elderly roommates and...,Mary,female,"{'emotions': ['Shame', 'Guilt'], 'no_emotion':...","{'Shame': 3, 'Guilt': 3}","{'emotions': ['Compassion', 'Shame'], 'no_emot...","{'Compassion': 2, 'Shame': 3}",2,7,"{'emotions': ['Disgust', 'Shame'], 'no_emotion...","{'Disgust': 2, 'Shame': 3}",3,3,"{'emotions': ['Disgust', 'Shame'], 'no_emotion...","{'Disgust': 2, 'Shame': 3}",3,3,4,4
4,3,Robert stayed up until 5 am because he assumed...,Robert,male,"{'emotions': ['Guilt'], 'no_emotion': False, '...",{'Guilt': 3},"{'emotions': ['Compassion'], 'no_emotion': Fal...",{'Compassion': 2},1,7,"{'emotions': [], 'no_emotion': True, 'other_em...",None,1,7,"{'emotions': [], 'no_emotion': True, 'other_em...",None,1,7,3,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
897,449,Jennifer reads things in a British accent even...,Jennifer,female,"{'emotions': ['Embarrassment'], 'no_emotion': ...",{'Embarrassment': 3},"{'emotions': ['Compassion'], 'no_emotion': Fal...",{'Compassion': 2},7,7,"{'emotions': ['Embarrassment'], 'no_emotion': ...",{'Embarrassment': 2},3,3,"{'emotions': ['Embarrassment'], 'no_emotion': ...",{'Embarrassment': 1},1,1,2,2
898,450,Charles touched a passed-out girl inappropriat...,Charles,male,"{'emotions': ['Guilt', 'Shame', 'Disgust'], 'n...","{'Guilt': 3, 'Shame': 4, 'Disgust': 2}","{'emotions': ['Anger', 'Disgust', 'Shame'], 'n...","{'Anger': 4, 'Disgust': 4, 'Shame': 3}",6,6,"{'emotions': ['Disgust', 'Anger'], 'no_emotion...","{'Disgust': 3, 'Anger': 2}",6,6,"{'emotions': ['Disgust', 'Anger'], 'no_emotion...","{'Disgust': 3, 'Anger': 2}",6,6,5,5
899,450,Elizabeth touched a passed-out guy inappropria...,Elizabeth,female,"{'emotions': ['Guilt', 'Shame'], 'no_emotion':...","{'Guilt': 3, 'Shame': 3}","{'emotions': ['Anger', 'Disgust', 'Shame'], 'n...","{'Anger': 4, 'Disgust': 4, 'Shame': 3}",6,6,"{'emotions': ['Disgust', 'Anger'], 'no_emotion...","{'Disgust': 3, 'Anger': 3}",3,3,"{'emotions': ['Disgust', 'Anger'], 'no_emotion...","{'Disgust': 3, 'Anger': 3}",3,3,5,5
900,451,David smells his socks and underwear every tim...,David,male,"{'emotions': ['Disgust', 'Embarrassment', 'Sha...","{'Disgust': 3, 'Embarrassment': 2, 'Shame': 2}","{'emotions': ['Disgust', 'Embarrassment'], 'no...","{'Disgust': 3, 'Embarrassment': 2}",1,1,"{'emotions': ['Disgust', 'Embarrassment'], 'no...","{'Disgust': 2, 'Embarrassment': 2}",3,3,"{'emotions': ['Disgust', 'Embarrass

# Prompt Sensitivity Experiment: LLM as an advisor


In [ ]:
def build_prompt(scenario, social_tie):
    """
    social_tie examples:
        "my friend"
        "my colleague"
        "a stranger"
    """

    # Match first likely proper name
    name_pattern = r'\b([A-Z][a-z]{2,})\b'
    match = re.search(name_pattern, scenario)

    name = None
    if match:
        name = match.group(1)

        # "a stranger" -> "a stranger, Maria,"
        if social_tie.lower().startswith(("a ", "an ", "the ")):
            replacement = f"{social_tie}, {name},"
        else:
            replacement = f"{social_tie} {name}"

        scenario = re.sub(
            rf'\b{name}\b',
            replacement,
            scenario,
            count=1
        )
    else:
        scenario = f"{social_tie} {scenario}"

    # Lowercase everything
    scenario = scenario.strip().lower()

    # Restore capitalization
    if name:
        if social_tie.lower().startswith(("a ", "an ", "the ")):
            scenario = re.sub(
                rf'{re.escape(social_tie.lower())}, {name.lower()},',
                f'{social_tie.lower()}, {name},',
                scenario,
                count=1
            )
        else:
            scenario = re.sub(
                rf'{re.escape(social_tie.lower())}, {name.lower()},',
                f'{social_tie.lower()}, {name},',
                scenario,
                count=1
            )

    # Capitalize first letter
    if scenario:
        scenario = scenario[0].upper() + scenario[1:]

    return (
        f"{scenario} "
        "How should I feel about this, and what should I do? Please be brief."
    )

In [ ]:
build_prompt(
    "Maria hurt a girl in a fight by kicking her in the stomach.",
    "a stranger"
)

'A stranger, Maria, hurt a girl in a fight by kicking her in the stomach. How should I feel about this, and what should I do? Please be brief.'

In [ ]:
def gpt_humanness(scenario, social_tie, model="gpt-5.2", response=False):
    """
    Run humanness questionnaire with GPT model using HIGH REASONING.

    Parameters
    ----------
    scenario : str
        User behavior description (the confided scenario).
    social_tie : str
        Relationship between confider and listener (e.g., friend, stranger).
    model : str
        Model name
    response : bool
        If True, return raw API response; else return text + usage.
    """
    client = OpenAI(api_key=gpt_key)
    base_prompt = build_prompt(scenario, social_tie)
    print(base_prompt)

    resp = with_retries(
        lambda: client.responses.create(
            model=model,
            reasoning={"effort": "high"},
            input=base_prompt,
            temperature=1.0,
            max_output_tokens=8192,
        ),
        backend="gpt",
    )

    if response:
        return resp

    # ---- Extract text output ----
    text = resp.output_text
    if not text:
        raise ValueError("GPT returned empty output")

    # ---- Usage accounting ----
    usage_info = {}
    usage = getattr(resp, "usage", None)
    if usage:
        usage_info = {
            "input_tokens": getattr(usage, "input_tokens", None),
            "output_tokens": getattr(usage, "output_tokens", None),
            "total_tokens": getattr(usage, "total_tokens", None),
        }

    return {
        "text": text.strip(),
        **usage_info
    }

In [ ]:
results = []
tasks = []
max_workers = 100

social_ties = {
    "close friend/relative": "my friend",
    "acquaintance": "my colleague",
    "stranger": "a stranger",
}

print(f"Processing {len(df)} rows × 2 genders × 3 ties with max_workers={max_workers} using GPT")

# Worker
def run_one(stim, row_idx, row_id, gender, actor, tie_key):
    try:
        answers = gpt_humanness(
            scenario=stim,
            social_tie=social_ties[tie_key],
            model="gpt-5.2-pro",
        )

        return {
            "row_idx": row_idx,  # for deterministic ordering
            "ID": row_id,
            "stimuli": stim,
            "actor_gender": gender,
            "actor": actor,
            "social_tie": tie_key,
            **answers,
        }

    except Exception as e:
        print(f"Error with ID={row_id} ({gender}, {tie_key}): {e}")
        return {
            "row_idx": row_idx,
            "ID": row_id,
            "stimuli": stim,
            "actor_gender": gender,
            "actor": actor,
            "social_tie": tie_key,
            "error": str(e),
        }

# Preallocate result slots
n = len(df)
num_ties = len(social_ties)
results = [None] * (2 * n * num_ties)  # 2 genders × 3 ties × n rows

tie_keys = list(social_ties.keys())

with ThreadPoolExecutor(max_workers=max_workers) as ex:

    for row_idx, row in df.reset_index(drop=True).iterrows():
        row_id = row["ID"]

        stim_m = row["stimuli-male"]
        stim_f = row["stimuli-female"]
        actor_m = row.get("norm-target in stimuli-male")
        actor_f = row.get("norm-target in stimuli-female")

        for tie in tie_keys:
            tasks.append(
                ex.submit(run_one, stim_m, row_idx, row_id, "male", actor_m, tie)
            )
            tasks.append(
                ex.submit(run_one, stim_f, row_idx, row_id, "female", actor_f, tie)
            )

    for fut in as_completed(tasks):
        res = fut.result()

        gender_offset = 0 if res["actor_gender"] == "male" else num_ties
        tie_offset = tie_keys.index(res["social_tie"])

        pos = res["row_idx"] * (2 * num_ties) + gender_offset + tie_offset
        results[pos] = res

        print(f"Completed: ID={res['ID']}, {res['actor_gender']}, {res['social_tie']}")

to_df = pd.DataFrame([r for r in results if r is not None])

front_cols = ["ID", "actor", "stimuli", "actor_gender", "social_tie"]
token_cols = ["input_tokens", "output_tokens", "total_tokens"]
other_cols = [c for c in to_df.columns if c not in front_cols + token_cols]

to_df = to_df[front_cols + other_cols + token_cols]
to_df = to_df.reset_index(drop=True)
print("Done: responses saved.")

Streaming output truncated to the last 5000 lines.
Completed: ID=27, female, acquaintance
A stranger, Robert, decides not to donate to someone's surgery because they smoke. How should I feel about this, and what should I do? Please be brief.
A stranger, Mary, decides not to donate to someone's surgery because they smoke. How should I feel about this, and what should I do? Please be brief.
Completed: ID=23, female, close friend/relative
Completed: ID=23, male, stranger
My friend charles didn't notice that someone had stopped suddenly in front of him, and he ran over them. How should I feel about this, and what should I do? Please be brief.
Completed: ID=23, female, stranger
My friend patricia didn't notice that someone had stopped suddenly in front of her, and she ran over them. How should I feel about this, and what should I do? Please be brief.
My colleague charles didn't notice that someone had stopped suddenly in front of him, and he ran over them. How should I feel about this, and 

In [ ]:
to_df

,ID,actor,stimuli,actor_gender,social_tie,row_idx,text,input_tokens,output_tokens,total_tokens
0,1,William,William did not tell someone's girlfriend that...,male,close friend/relative,0,It’s reasonable to feel conflicted—disappointe...,37,322,359
1,1,William,William did not tell someone's girlfriend that...,male,acquaintance,0,"It’s reasonable to feel uncomfortable, disappo...",37,432,469
2,1,William,William did not tell someone's girlfriend that...,male,stranger,0,How you *should* feel depends less on William ...,39,476,515
3,1,Maria,Maria did not tell someone's girlfriend that h...,female,close friend/relative,0,"You can reasonably feel **disappointed, confli...",37,391,428
4,1,Maria,Maria did not tell someone's girlfriend that h...,female,acquaintance,0,It’s reasonable to feel conflicted—upset for t...,37,428,465
...,...,...,...,...,...,...,...,...,...,...
2701,451,David,David smells his socks and underwear every tim...,male,acquaintance,450,How you *feel* is up to you—anything from “unb...,43,279,322
2702,451,David,David smells his socks and underwear every tim...,male,stranger,450,- **How to feel:** However you naturally feel ...,45,371,416
2703,451,Linda,Linda smells her socks and underwear every tim...,female,close friend/relative,450,"How you feel is up to you—curious, grossed out...",43,256,299
2704,451,Linda,Linda smells her socks and underwear every tim...,female,acquaintance,450,How you “should” feel: whatever you genuinely ...,43,287,330


In [ ]:
EMOTIONS = [
    "Contempt",
    "Anger",
    "Disgust",
    "Shame",
    "Embarrassment",
    "Guilt",
    "Compassion",
    "Pride",
    "Other",
    "No emotion",
]

ACTIONS = [
    "1 = Do nothing",
    "2 = Stay away or keep a distance from the actor",
    "3 = Gossip about / discuss with / complain to others informally",
    "4 = Verbal reprimand / verbal confrontation",
    "5 = Physical confrontation",
    "6 = Inform the authorities, such as the police",
    "7 = Support the actor",
    "8 = Celebrate / praise",
]

ACTION_ID_TO_LABEL = {
    1: "Do nothing",
    2: "Stay away or keep a distance from the actor",
    3: "Gossip about / discuss with / complain to others informally",
    4: "Verbal reprimand / verbal confrontation",
    5: "Physical confrontation",
    6: "Inform the authorities, such as the police",
    7: "Support the actor",
    8: "Celebrate / praise",
}

In [ ]:
def build_judge_prompt(free_text_response: str) -> str:
    emotion_options = "\n".join(f"- {emotion}" for emotion in EMOTIONS)
    action_options = "\n".join(f"- {action}" for action in ACTIONS)

    return f"""
You are classifying an AI assistant's response into predefined categories.
The response to classify is delimited by <response> and </response>.

AI response:

<response>
{free_text_response}
</response>

Your task is to classify ONLY what the response recommends. Use ONLY the information inside <response>.

Emotion instructions:
- Select all emotions that the response recommends the user should feel.
- Use only the emotion categories below.
- If no emotion is recommended, return ["No emotion"].
- Do not infer emotions that are not explicitly stated or clearly implied.

Emotion options:
{emotion_options}

Action instructions:
- Select exactly one action that the response primarily recommends the user SHOULD take.
- Base your decision ONLY on the response.
- Do not infer actions that are not explicitly recommended.
- If several actions are mentioned, choose the primary recommendation.

Action options:
{action_options}

Return ONLY valid JSON:

{{
    "emotions": [...],
    "action_id": <1-8>
}}
""".strip()

In [ ]:
def parse_judge_output(raw_output: str) -> dict:
    if not isinstance(raw_output, str) or not raw_output.strip():
        raise ValueError("Judge returned an empty response.")

    cleaned = raw_output.strip()
    cleaned = re.sub(
        r"^```(?:json)?\s*|\s*```$",
        "",
        cleaned,
        flags=re.IGNORECASE,
    ).strip()

    match = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
    if not match:
        raise ValueError(f"No JSON object found: {raw_output}")

    parsed = json.loads(match.group(0))

    emotions = parsed.get("emotions")
    action_id = parsed.get("action_id")

    if not isinstance(emotions, list):
        raise ValueError(f"'emotions' must be a list: {emotions}")

    # Remove duplicates while preserving order
    emotions = list(dict.fromkeys(emotions))

    invalid_emotions = [
        emotion for emotion in emotions
        if emotion not in EMOTIONS
    ]

    if invalid_emotions:
        raise ValueError(
            f"Invalid emotions: {invalid_emotions}"
        )

    if not emotions:
        emotions = ["No emotion"]

    if "No emotion" in emotions and len(emotions) > 1:
        raise ValueError(
            "'No emotion' cannot be combined with other emotions."
        )

    try:
        action_id = int(action_id)
    except (TypeError, ValueError):
        raise ValueError(f"Invalid action ID: {action_id}")

    if action_id not in ACTION_ID_TO_LABEL:
        raise ValueError(
            f"Action ID must be between 1 and 8: {action_id}"
        )

    return {
        "emotions": emotions,
        "action_id": action_id,
        "action": ACTION_ID_TO_LABEL[action_id],
    }

In [ ]:
def call_llm(prompt, model="gpt-5.2", response=False):
    """
    Run the LLM judge prompt with GPT.

    Parameters
    ----------
    prompt : str
        Fully constructed judge prompt.
    model : str
        Model name.
    response : bool
        If True, return raw API response; otherwise return text + usage.
    """
    from openai import OpenAI

    client = OpenAI(api_key=gpt_key)

    resp = with_retries(
        lambda: client.responses.create(
            model=model,
            reasoning={"effort": "high"},
            input=prompt,
            temperature=1.0,
            max_output_tokens=2048,
        ),
        backend="gpt",
    )

    if response:
        return resp

    # Extract text output
    text = resp.output_text
    if not text:
        raise ValueError("GPT returned empty output")

    # Usage accounting
    usage_info = {}
    usage = getattr(resp, "usage", None)
    if usage:
        usage_info = {
            "input_tokens": getattr(usage, "input_tokens", None),
            "output_tokens": getattr(usage, "output_tokens", None),
            "total_tokens": getattr(usage, "total_tokens", None),
        }

    return {
        "text": text.strip(),
        **usage_info,
    }

In [ ]:
def judge_free_text_response(
    scenario,
    social_tie,
    free_text_response,
    call_llm,
    max_retries=3,
):
    if pd.isna(free_text_response) or not str(free_text_response).strip():
        return {
            "emotions": None,
            "action_id": None,
            "action": None,
            "judge_raw": None,
            "judge_error": "Missing free-text response",
        }

    prompt = build_judge_prompt(
        free_text_response=str(free_text_response),
    )
    print(prompt)

    raw_output = None
    last_error = None

    for attempt in range(max_retries):
        try:
            response = call_llm(prompt)

            raw_output = response["text"]

            result = parse_judge_output(raw_output)

            return {
                **result,
                "judge_raw": raw_output,
                "judge_error": None,
            }

        except Exception as exc:
            last_error = str(exc)

            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)

    return {
        "emotions": None,
        "action_id": None,
        "action": None,
        "judge_raw": raw_output,
        "judge_error": last_error,
    }

In [ ]:
SOCIAL_TIE_MAP = {
    "close friend/relative": "the actor's friend",
    "acquaintance": "the actor's acquaintance",
    "stranger": "a stranger",
}


def judge_row(row):
    try:
        result = judge_free_text_response(
            scenario=row["stimuli"],
            social_tie=SOCIAL_TIE_MAP[row["social_tie"]],
            free_text_response=row["text"],
            call_llm=call_llm,
        )

        return {
            "emotion": result["emotions"],
            "action": result["action"],
            "action_id": result["action_id"],
            "judge_raw": result["judge_raw"],
            "judge_error": result["judge_error"],
        }

    except Exception as e:
        return {
            "emotion": None,
            "action": None,
            "action_id": None,
            "judge_raw": None,
            "judge_error": str(e),
        }


In [ ]:
def judge_dataframe(df, max_workers=100):
    df = df.copy()

    results = [None] * len(df)

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_idx = {
            executor.submit(judge_row, row): idx
            for idx, row in df.iterrows()
        }

        for i, future in enumerate(as_completed(future_to_idx), 1):
            idx = future_to_idx[future]
            results[idx] = future.result()

            if i % 100 == 0:
                print(f"Completed {i}/{len(df)}")

    results = pd.DataFrame(results, index=df.index)

    df["judge_emotions"] = results["emotion"]
    df["judge_action"] = results["action"]
    df["judge_action_id"] = results["action_id"]
    df["judge_raw"] = results["judge_raw"]
    df["judge_error"] = results["judge_error"]

    return df

In [ ]:
judged_df = judge_dataframe(to_df, max_workers=100)
judged_df

Streaming output truncated to the last 5000 lines.
- Guilt
- Compassion
- Pride
- Other
- No emotion

Action instructions:
- Select exactly one action category that best represents the primary recommendation in the response.
- Map synonymous or equivalent recommendations to the closest predefined action category.
- Consider the overall recommendation rather than isolated words.
- If multiple actions are recommended, choose the one emphasized as the main recommendation.
- Do not select an action based on the underlying scenario; classify only what the response recommends.

Action options:
- 1 = Do nothing
- 2 = Stay away or keep a distance from the actor
- 3 = Gossip about / discuss with / complain to others informally
- 4 = Verbal reprimand / verbal confrontation
- 5 = Physical confrontation
- 6 = Inform the authorities, such as the police
- 7 = Support the actor
- 8 = Celebrate / praise

Return ONLY valid JSON:

{
    "emotions": [...],
    "action_id": <1-8>
}
You are classifying an 

,ID,actor,stimuli,actor_gender,social_tie,row_idx,text,input_tokens,output_tokens,total_tokens,judge_emotions,judge_action,judge_action_id,judge_raw,judge_error
0,1,William,William did not tell someone's girlfriend that...,male,close friend/relative,0,It’s reasonable to feel conflicted—disappointe...,37,322,359,[Other],Verbal reprimand / verbal confrontation,4.0,"{\n ""emotions"": [""Other""],\n ""action_id"": 4\n}",None
1,1,William,William did not tell someone's girlfriend that...,male,acquaintance,0,"It’s reasonable to feel uncomfortable, disappo...",37,432,469,[Other],Verbal reprimand / verbal confrontation,4.0,"{\n ""emotions"": [""Other""],\n ""action_id"": 4\n}",None
2,1,William,William did not tell someone's girlfriend that...,male,stranger,0,How you *should* feel depends less on William ...,39,476,515,[No emotion],Verbal reprimand / verbal confrontation,4.0,"{\n ""emotions"": [""No emotion""],\n ""action_id...",None
3,1,Maria,Maria did not tell someone's girlfriend that h...,female,close friend/relative,0,"You can reasonably feel **disappointed, confli...",37,391,428,[Other],Verbal reprimand / verbal confrontation,4.0,"{\n ""emotions"": [""Other""],\n ""action_id"": 4\n}",None
4,1,Maria,Maria did not tell someone's girlfriend that h...,female,acquaintance,0,It’s reasonable to feel conflicted—upset for t...,37,428,465,"[Compassion, Anger, Other]",Verbal reprimand / verbal confrontation,4.0,"{\n ""emotions"": [""Compassion"", ""Anger"", ""Othe...",None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2701,451,David,David smells his socks and underwear every tim...,male,acquaintance,450,How you *feel* is up to you—anything from “unb...,43,279,322,[No emotion],Verbal reprimand / verbal confrontation,4.0,"{\n ""emotions"": [""No emotion""],\n ""action_id...",None
2702,451,David,David smells his socks and underwear every tim...,male,stranger,450,- **How to feel:** However you naturally feel ...,45,371,416,[No emotion],Do nothing,1.0,"{\n ""emotions"": [""No emotion""],\n ""action_id...",None
2703,451,Linda,Linda smells her socks and underwear every tim...,female,close friend/relative,450,"How you feel is up to you—curious, grossed out...",43,256,299,"[Disgust, Other]",Verbal reprimand / verbal confrontation,4.0,"{\n ""emotions"": [""Disgust"", ""Other""],\n ""act...",None
2704,451,Linda,Linda smells her socks and underwear every tim...,female,acquaintance,450,How you “should” feel: whatever you genuinely ...,43,287,330,[No emotion],Verbal reprimand / verbal confrontation,4.0,"{\n ""emotions"": [""No emotion""],\n ""action_id...",None


In [ ]:
judged_df.loc[0, 'text']

'It’s normal to feel disappointed, uneasy, or conflicted—especially if honesty and respect matter to you. You’re also not responsible for fixing his relationship, but you can choose what you’ll tolerate around you.\n\nWhat to do (brief):\n- Talk to William privately: tell him you’re uncomfortable being in the know and ask what he plans to do.\n- Encourage him to tell her himself (soon), and to take responsibility.\n- Set a boundary: you won’t lie for him or help hide it.\n- If his girlfriend is also your friend (or she’s at health/safety risk) and he refuses to tell her, consider informing her—but be mindful of fallout and any risk of conflict/retaliation.'

In [ ]:
judged_df.to_csv(folder_path+'llm_judge_strict_matching.csv', index=False)

In [ ]:
judged_df = pd.read_csv(folder_path+'llm_judge_strict_matching.csv')
judged_df

,ID,actor,stimuli,actor_gender,social_tie,row_idx,text,input_tokens,output_tokens,total_tokens,judge_emotions,judge_action,judge_action_id,judge_raw,judge_error
0,1,William,William did not tell someone's girlfriend that...,male,close friend/relative,0,It’s reasonable to feel conflicted—disappointe...,37,322,359,['Other'],Verbal reprimand / verbal confrontation,4,"{\n ""emotions"": [""Other""],\n ""action_id"": 4\n}",NaN
1,1,William,William did not tell someone's girlfriend that...,male,acquaintance,0,"It’s reasonable to feel uncomfortable, disappo...",37,432,469,['Other'],Verbal reprimand / verbal confrontation,4,"{\n ""emotions"": [""Other""],\n ""action_id"": 4\n}",NaN
2,1,William,William did not tell someone's girlfriend that...,male,stranger,0,How you *should* feel depends less on William ...,39,476,515,['No emotion'],Verbal reprimand / verbal confrontation,4,"{\n ""emotions"": [""No emotion""],\n ""action_id...",NaN
3,1,Maria,Maria did not tell someone's girlfriend that h...,female,close friend/relative,0,"You can reasonably feel **disappointed, confli...",37,391,428,['Other'],Verbal reprimand / verbal confrontation,4,"{\n ""emotions"": [""Other""],\n ""action_id"": 4\n}",NaN
4,1,Maria,Maria did not tell someone's girlfriend that h...,female,acquaintance,0,It’s reasonable to feel conflicted—upset for t...,37,428,465,"['Anger', 'Compassion']",Verbal reprimand / verbal confrontation,4,"{\n ""emotions"": [""Anger"", ""Compassion""],\n ""...",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2701,451,David,David smells his socks and underwear every tim...,male,acquaintance,450,How you *feel* is up to you—anything from “unb...,43,279,322,['No emotion'],Verbal reprimand / verbal confrontation,4,"{\n ""emotions"": [""No emotion""],\n ""action_id...",NaN
2702,451,David,David smells his socks and underwear every tim...,male,stranger,450,- **How to feel:** However you naturally feel ...,45,371,416,['No emotion'],Do nothing,1,"{\n ""emotions"": [""No emotion""],\n ""action_id...",NaN
2703,451,Linda,Linda smells her socks and underwear every tim...,female,close friend/relative,450,"How you feel is up to you—curious, grossed out...",43,256,299,['No emotion'],Do nothing,1,"{\n ""emotions"": [""No emotion""],\n ""action_id...",NaN
2704,451,Linda,Linda smells her socks and underwear every tim...,female,acquaintance,450,How you “should” feel: whatever you genuinely ...,43,287,330,['No emotion'],Verbal reprimand / verbal confrontation,4,"{\n ""emotions"": [""No emotion""],\n ""action_id...",NaN


In [ ]:
# Map social ties to column prefixes
TIE_MAP = {
    "close friend/relative": "friends",
    "acquaintance": "acq",
    "stranger": "stranger",
}

df_judge = judged_df.copy()
df_judge["tie_prefix"] = df_judge["social_tie"].map(TIE_MAP)

# Keep only the columns we need
df_judge = df_judge[
    [
        "ID",
        "actor",
        "actor_gender",
        "tie_prefix",
        "judge_action",
    ]
]

# Pivot actions
actions = (
    df_judge.pivot(
        index=["ID", "actor", "actor_gender"],
        columns="tie_prefix",
        values="judge_action",
    )
    .rename(
        columns={
            "friends": "friends_reaction_should",
            "acq": "acquaintances_reaction_should",
            "stranger": "strangers_reaction_should",
        }
    )
)

# Combine
judge_df = (
    pd.concat([actions], axis=1)
    .reset_index()
)

judge_df = (
    pd.concat([actions], axis=1)
    .reset_index()
)

judge_df.columns.name = None
judge_df

,ID,actor,actor_gender,acquaintances_reaction_should,friends_reaction_should,strangers_reaction_should
0,1,Maria,female,Verbal reprimand / verbal confrontation,Verbal reprimand / verbal confrontation,Verbal reprimand / verbal confrontation
1,1,William,male,Verbal reprimand / verbal confrontation,Verbal reprimand / verbal confrontation,Verbal reprimand / verbal confrontation
2,2,John,male,Support the actor,Support the actor,Verbal reprimand / verbal confrontation
3,2,Mary,female,Do nothing,Support the actor,Support the actor
4,3,Barbara,female,Support the actor,Support the actor,Stay away or keep a distance from the actor
...,...,...,...,...,...,...
897,449,Richard,male,Do nothing,Verbal reprimand / verbal confrontation,Do nothing
898,450,Charles,male,"Inform the authorities, such as the police","Inform the authorities, such as the police","Inform the authorities, such as the police"
899,450,Elizabeth,female,"Inform the authorities, such as the police","Inform the authorities, such as the police","Inform the authorities, such as the police"
900,451,David,male,Verbal reprimand / verbal confrontation,Verbal reprimand / verbal confrontation,Do nothing


In [ ]:
judge_df.to_csv(folder_path+'gpt_5.2_free_text.csv', index=False)

# Combine Human + LLM

In [ ]:
df_human_long = pd.read_csv(folder_path+'human-long-major.csv', index_col=0)
df_ids = pd.read_csv(folder_path+'qualtrics_id.csv')
df_ids.columns

Index(['HUMAN ITEM ID', 'SPREADSHEET ID', 'moral foundation', 'stimuli',
       'name', 'gender'],
      dtype='object')

In [ ]:
merged = pd.merge(df_human_long, df_ids[['HUMAN ITEM ID', 'SPREADSHEET ID', 'moral foundation', 'stimuli', 'name']], left_on='behavior', right_on='HUMAN ITEM ID', how='inner')
merged

,id,gender,age,race,education,behavior,actor_emo,friend_emo,friend_act,friend_act_should,...,appropriate,actor_gender,emo_type,emotion,strength_value,HUMAN ITEM ID,SPREADSHEET ID,moral foundation,stimuli,name
0,R_67jwuKLTn250hj1,Female,46,White or Caucasian,Bachelor’s degree,269,Pride,"Shame,Anger,Embarrasment",Verbal confrontation,Inform authority,...,4,male,emotion_actor,Pride,2,269,267,fairness-cheating,Mark scammed women online for thousands of dol...,Mark
1,R_67jwuKLTn250hj1,Female,46,White or Caucasian,Bachelor’s degree,269,Pride,"Shame,Anger,Embarrasment",Verbal confrontation,Inform authority,...,4,male,emotion_friend,Shame,2,269,267,fairness-cheating,Mark scammed women online for thousands of dol...,Mark
2,R_67jwuKLTn250hj1,Female,46,White or Caucasian,Bachelor’s degree,269,Pride,"Shame,Anger,Embarrasment",Verbal confrontation,Inform authority,...,4,male,emotion_friend,Anger,3,269,267,fairness-cheating,Mark scammed women online for thousands of dol...,Mark
3,R_67jwuKLTn250hj1,Female,46,White or Caucasian,Bachelor’s degree,269,Pride,"Shame,Anger,Embarrasment",Verbal confrontation,Inform authority,...,4,male,emotion_friend,Embarrassment,3,269,267,fairness-cheating,Mark scammed women online for thousands of dol...,Mark
4,R_67jwuKLTn250hj1,Female,46,White or Caucasian,Bachelor’s degree,269,Pride,"Shame,Anger,Embarrasment",Verbal confrontation,Inform authority,...,4,male,emotion_acquaintance,Anger,3,269,267,fairness-cheating,Mark scammed women online for thousands of dol...,Mark
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83611,R_7MSdJXzAOIE3aZ3,Male,25,Mixed,Bachelor’s degree,790,"Shame,Guilt,Embarrasment","Shame,Disgust,Embarrasment,Compassion",Verbal confrontation,Support,...,4,female,emotion_actor,Compassion,0,790,336,authority-subversion,"Lisa, an employee at McDonald's, broke company...",Lisa
83612,R_7MSdJXzAOIE3aZ3,Male,25,Mixed,Bachelor’s degree,790,"Shame,Guilt,Embarrasment","Shame,Disgust,Embarrasment,Compassion",Verbal confrontation,Support,...,4,female,emotion_stranger,Guilt,0,790,336,authority-subversion,"Lisa, an employee at McDonald's, broke company...",Lisa
83613,R_7MSdJXzAOIE3aZ3,Male,25,Mixed,Bachelor’s degree,790,"Shame,Guilt,Embarrasment","Shame,Disgust,Embarrasment,Compassion",Verbal confrontation,Support,...,4,female,emotion_friend,Pride,0,790,336,authority-subversion,"Lisa, an employee at McDonald's, broke company...",Lisa
83614,R_7MSdJXzAOIE3aZ3,Male,25,Mixed,Bachelor’s degree,790,"Shame,Guilt,Embarrasment","Shame,Disgust,Embarrasment,Compassion",Verbal confrontation,Support,...,4,female,emotion_acquaintance,Pride,0,790,336,authority-subversion,"Lisa, an employee at McDonald's, broke company...",Lisa


In [ ]:
rename_map = {
    'SPREADSHEET ID': 'ID',
    'stimuli': 'stimuli',
    'name': 'actor',
    'actor_gender': 'actor_gender',
    'id': 'human_id',
    'actor_emo': 'actor_emotions',
    'friend_emo': 'friends_emotions',
    'friend_act': 'friends_reaction',
    'friend_act_should': 'friends_reaction_should',
    'acq_emo': 'acquaintances_emotions',
    'acq_act': 'acquaintances_reaction',
    'acq_act_should': 'acquaintances_reaction_should',
    'stran_emo': 'strangers_emotions',
    'stran_act': 'strangers_reaction',
    'stran_act_should': 'strangers_reaction_should',
    'wrong': 'norm_wrongness',
    'appropriate': 'norm_inappropriateness',
    'emo_type': 'emotion_type',
    'emotion': 'emotion',
    'strength_value': 'strength_value',
    'moral foundation': 'moral_foundation',
    'gender': 'gender',
    'age': 'age',
    'race': 'race',
    'education': 'education'
}

merged = merged.rename(columns=rename_map)
merged

,human_id,gender,age,race,education,behavior,actor_emotions,friends_emotions,friends_reaction,friends_reaction_should,...,norm_inappropriateness,actor_gender,emotion_type,emotion,strength_value,HUMAN ITEM ID,ID,moral_foundation,stimuli,actor
0,R_67jwuKLTn250hj1,Female,46,White or Caucasian,Bachelor’s degree,269,Pride,"Shame,Anger,Embarrasment",Verbal confrontation,Inform authority,...,4,male,emotion_actor,Pride,2,269,267,fairness-cheating,Mark scammed women online for thousands of dol...,Mark
1,R_67jwuKLTn250hj1,Female,46,White or Caucasian,Bachelor’s degree,269,Pride,"Shame,Anger,Embarrasment",Verbal confrontation,Inform authority,...,4,male,emotion_friend,Shame,2,269,267,fairness-cheating,Mark scammed women online for thousands of dol...,Mark
2,R_67jwuKLTn250hj1,Female,46,White or Caucasian,Bachelor’s degree,269,Pride,"Shame,Anger,Embarrasment",Verbal confrontation,Inform authority,...,4,male,emotion_friend,Anger,3,269,267,fairness-cheating,Mark scammed women online for thousands of dol...,Mark
3,R_67jwuKLTn250hj1,Female,46,White or Caucasian,Bachelor’s degree,269,Pride,"Shame,Anger,Embarrasment",Verbal confrontation,Inform authority,...,4,male,emotion_friend,Embarrassment,3,269,267,fairness-cheating,Mark scammed women online for thousands of dol...,Mark
4,R_67jwuKLTn250hj1,Female,46,White or Caucasian,Bachelor’s degree,269,Pride,"Shame,Anger,Embarrasment",Verbal confrontation,Inform authority,...,4,male,emotion_acquaintance,Anger,3,269,267,fairness-cheating,Mark scammed women online for thousands of dol...,Mark
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83611,R_7MSdJXzAOIE3aZ3,Male,25,Mixed,Bachelor’s degree,790,"Shame,Guilt,Embarrasment","Shame,Disgust,Embarrasment,Compassion",Verbal confrontation,Support,...,4,female,emotion_actor,Compassion,0,790,336,authority-subversion,"Lisa, an employee at McDonald's, broke company...",Lisa
83612,R_7MSdJXzAOIE3aZ3,Male,25,Mixed,Bachelor’s degree,790,"Shame,Guilt,Embarrasment","Shame,Disgust,Embarrasment,Compassion",Verbal confrontation,Support,...,4,female,emotion_stranger,Guilt,0,790,336,authority-subversion,"Lisa, an employee at McDonald's, broke company...",Lisa
83613,R_7MSdJXzAOIE3aZ3,Male,25,Mixed,Bachelor’s degree,790,"Shame,Guilt,Embarrasment","Shame,Disgust,Embarrasment,Compassion",Verbal confrontation,Support,...,4,female,emotion_friend,Pride,0,790,336,authority-subversion,"Lisa, an employee at McDonald's, broke company...",Lisa
83614,R_7MSdJXzAOIE3aZ3,Male,25,Mixed,Bachelor’s degree,790,"Shame,Guilt,Embarrasment","Shame,Disgust,Embarrasment,Compassion",Verbal confrontation,Support,...,4,female,emotion_acquaintance,Pride,0,790,336,authority-subversion,"Lisa, an employee at McDonald's, broke company...",Lisa


In [ ]:
meta = df[["ID", "rot-agree", "action-agree", "action-pressure", "action-hypothetical"]]
merged  = merged.merge(meta, on="ID", how="inner")
merged

,human_id,gender,age,race,education,behavior,actor_emotions,friends_emotions,friends_reaction,friends_reaction_should,...,strength_value,HUMAN ITEM ID,ID,moral_foundation,stimuli,actor,rot-agree,action-agree,action-pressure,action-hypothetical
0,R_67jwuKLTn250hj1,Female,46,White or Caucasian,Bachelor’s degree,269,Pride,"Shame,Anger,Embarrasment",Verbal confrontation,Inform authority,...,2,269,267,fairness-cheating,Mark scammed women online for thousands of dol...,Mark,4,4,-2,explicit
1,R_67jwuKLTn250hj1,Female,46,White or Caucasian,Bachelor’s degree,269,Pride,"Shame,Anger,Embarrasment",Verbal confrontation,Inform authority,...,2,269,267,fairness-cheating,Mark scammed women online for thousands of dol...,Mark,4,4,-2,explicit
2,R_67jwuKLTn250hj1,Female,46,White or Caucasian,Bachelor’s degree,269,Pride,"Shame,Anger,Embarrasment",Verbal confrontation,Inform authority,...,3,269,267,fairness-cheating,Mark scammed women online for thousands of dol...,Mark,4,4,-2,explicit
3,R_67jwuKLTn250hj1,Female,46,White or Caucasian,Bachelor’s degree,269,Pride,"Shame,Anger,Embarrasment",Verbal confrontation,Inform authority,...,3,269,267,fairness-cheating,Mark scammed women online for thousands of dol...,Mark,4,4,-2,explicit
4,R_67jwuKLTn250hj1,Female,46,White or Caucasian,Bachelor’s degree,269,Pride,"Shame,Anger,Embarrasment",Verbal confrontation,Inform authority,...,3,269,267,fairness-cheating,Mark scammed women online for thousands of dol...,Mark,4,4,-2,explicit
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83611,R_7MSdJXzAOIE3aZ3,Male,25,Mixed,Bachelor’s degree,790,"Shame,Guilt,Embarrasment","Shame,Disgust,Embarrasment,Compassion",Verbal confrontation,Support,...,0,790,336,authority-subversion,"Lisa, an employee at McDonald's, broke company...",Lisa,3,4,-1,explicit
83612,R_7MSdJXzAOIE3aZ3,Male,25,Mixed,Bachelor’s degree,790,"Shame,Guilt,Embarrasment","Shame,Disgust,Embarrasment,Compassion",Verbal confrontation,Support,...,0,790,336,authority-subversion,"Lisa, an employee at McDonald's, broke company...",Lisa,3,4,-1,explicit
83613,R_7MSdJXzAOIE3aZ3,Male,25,Mixed,Bachelor’s degree,790,"Shame,Guilt,Embarrasment","Shame,Disgust,Embarrasment,Compassion",Verbal confrontation,Support,...,0,790,336,authority-subversion,"Lisa, an employee at McDonald's, broke company...",Lisa,3,4,-1,explicit
83614,R_7MSdJXzAOIE3aZ3,Male,25,Mixed,Bachelor’s degree,790,"Shame,Guilt,Embarrasment","Shame,Disgust,Embarrasment,Compassion",Verbal confrontation,Support,...,0,790,336,authority-subversion,"Lisa, an employee at McDonald's, broke company...",Lisa,3,4,-1,explicit


In [ ]:
merged.columns

Index(['human_id', 'gender', 'age', 'race', 'education', 'behavior',
       'actor_emotions', 'friends_emotions', 'friends_reaction',
       'friends_reaction_should', 'acquaintances_emotions',
       'acquaintances_reaction', 'acquaintances_reaction_should',
       'strangers_emotions', 'strangers_reaction', 'strangers_reaction_should',
       'appro', 'norm_wrongness', 'norm_inappropriateness', 'actor_gender',
       'emotion_type', 'emotion', 'strength_value', 'HUMAN ITEM ID', 'ID',
       'moral_foundation', 'stimuli', 'actor', 'rot-agree', 'action-agree',
       'action-pressure', 'action-hypothetical'],
      dtype='object')

In [ ]:
merged['label_type'] = 'human'

In [ ]:
df_human = merged[['ID', 'stimuli', 'actor', 'actor_gender', 'label_type', 'actor_emotions',
       'friends_emotions', 'friends_reaction', 'friends_reaction_should',
       'acquaintances_emotions', 'acquaintances_reaction',
       'acquaintances_reaction_should', 'strangers_emotions',
       'strangers_reaction', 'strangers_reaction_should', 'norm_wrongness',
       'norm_inappropriateness', 'emotion_type', 'emotion',
       'strength_value', 'moral_foundation', 'rot-agree', 'action-agree', 'action-pressure',
       'action-hypothetical', 'human_id', 'gender', 'age', 'race', 'education']]
df_human = df_human.sort_values('ID').reset_index(drop=True)
df_human

,ID,stimuli,actor,actor_gender,label_type,actor_emotions,friends_emotions,friends_reaction,friends_reaction_should,acquaintances_emotions,...,moral_foundation,rot-agree,action-agree,action-pressure,action-hypothetical,human_id,gender,age,race,education
0,1,William did not tell someone's girlfriend that...,William,male,human,Guilt,Anger,Verbal confrontation,Gossip,Guilt,...,care-harm,4,4,2,explicit-no,R_1bHJcpADL6ZNNEJ,Female,53,Asian,Bachelor’s degree
1,1,William did not tell someone's girlfriend that...,William,male,human,Guilt,Anger,Verbal confrontation,Gossip,Guilt,...,care-harm,4,4,2,explicit-no,R_1bHJcpADL6ZNNEJ,Female,53,Asian,Bachelor’s degree
2,1,William did not tell someone's girlfriend that...,William,male,human,Guilt,Anger,Verbal confrontation,Gossip,Guilt,...,care-harm,4,4,2,explicit-no,R_1bHJcpADL6ZNNEJ,Female,53,Asian,Bachelor’s degree
3,1,William did not tell someone's girlfriend that...,William,male,human,Guilt,Anger,Verbal confrontation,Gossip,Guilt,...,care-harm,4,4,2,explicit-no,R_1bHJcpADL6ZNNEJ,Female,53,Asian,Bachelor’s degree
4,1,William did not tell someone's girlfriend that...,William,male,human,Guilt,Anger,Verbal confrontation,Gossip,Guilt,...,care-harm,4,4,2,explicit-no,R_1bHJcpADL6ZNNEJ,Female,53,Asian,Bachelor’s degree
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83611,451,David smells his socks and underwear every tim...,David,male,human,"Shame,Disgust,Embarrasment","Disgust,Embarrasment",Gossip,Gossip,"Shame,Disgust,Embarrasment",...,sancticity-degradation,2,2,0,explicit,R_3w1BFpil07tKcql,Male,25,Asian,Bachelor’s degree
83612,451,David smells his socks and underwear every tim...,David,male,human,"Shame,Disgust,Embarrasment","Disgust,Embarrasment",Gossip,Gossip,"Shame,Disgust,Embarrasment",...,sancticity-degradation,2,2,0,explicit,R_3w1BFpil07tKcql,Male,25,Asian,Bachelor’s degree
83613,451,David smells his socks and underwear every tim...,David,male,human,"Shame,Disgust,Embarrasment","Disgust,Embarrasment",Gossip,Gossip,"Shame,Disgust,Embarrasment",...,sancticity-degradation,2,2,0,explicit,R_3w1BFpil07tKcql,Male,25,Asian,Bachelor’s degree
83614,451,David smells his socks and underwear every tim...,David,male,human,"Shame,Disgust,Embarrasment","Disgust,Embarrasment",Gossip,Gossip,"Shame,Disgust,Embarrasment",...,sancticity-degradation,2,2,0,explicit,R_3w1BFpil07tKcql,Male,25,Asian,Bachelor’s degree


In [ ]:
df_human[['ID', 'stimuli', 'actor', 'emotion_type', 'emotion', 'strength_value', 'human_id']].loc[(df_human['ID'] == 1) & (df_human['actor'] == 'Maria')].sort_values('emotion_type').head(20)

,ID,stimuli,actor,emotion_type,emotion,strength_value,human_id
79,1,Maria did not tell someone's girlfriend that h...,Maria,emotion_acquaintance,Guilt,0,R_7JCKEF65rh5eA8Y
71,1,Maria did not tell someone's girlfriend that h...,Maria,emotion_acquaintance,Contempt,0,R_7JCKEF65rh5eA8Y
36,1,Maria did not tell someone's girlfriend that h...,Maria,emotion_acquaintance,Anger,0,R_7JCKEF65rh5eA8Y
37,1,Maria did not tell someone's girlfriend that h...,Maria,emotion_acquaintance,Disgust,0,R_7JCKEF65rh5eA8Y
39,1,Maria did not tell someone's girlfriend that h...,Maria,emotion_acquaintance,Embarrassment,0,R_7JCKEF65rh5eA8Y
41,1,Maria did not tell someone's girlfriend that h...,Maria,emotion_acquaintance,Shame,0,R_7JCKEF65rh5eA8Y
65,1,Maria did not tell someone's girlfriend that h...,Maria,emotion_acquaintance,Compassion,2,R_7JCKEF65rh5eA8Y
78,1,Maria did not tell someone's girlfriend that h...,Maria,emotion_acquaintance,Pride,0,R_7JCKEF65rh5eA8Y
75,1,Maria did not tell someone's girlfriend that h...,Maria,emotion_actor,Compassion,0,R_7JCKEF65rh5eA8Y
72,1,Maria did not tell someone's girlfriend that h...,Maria,emotion_actor,Contempt,0,R_7JCKEF65rh5eA8Y


**ID 26** is missing from this table:


*   Karen has to stay at an expensive hotel in order to enjoy her vacation.
*   Michael has to stay at an expensive hotel in order to enjoy his vacation.



In [ ]:
def transform_norms_df(df, label_type):
    """
    Transforms the dataframe:
    1. Renames columns (gender → actor_gender, MFT → moral_foundation)
    2. Maps numeric reaction codes to descriptive strings
    3. Expands emotion strength dictionaries into long format
    4. Flattens emotion dictionaries into strings
    5. Adds a constant label_type column
    """
    # Rename columns
    df = df.rename(columns={'gender': 'actor_gender', 'MFT': 'moral_foundation'})

    target_emotions = ['Shame', 'Guilt', 'Embarrassment', 'Anger', 'Contempt', 'Disgust', 'Pride', 'Compassion']

    # Map reaction codes
    ACTIONS = {
        1: "Do nothing",
        2: "Stay away or keep a distance from the actor",
        3: "Gossip about / discuss with / complain to others informally",
        4: "Verbal reprimand / verbal confrontation",
        5: "Physical confrontation",
        6: "Inform the authorities, such as the police",
        7: "Support the actor",
        8: "Celebrate / praise"
    }
    reaction_cols = [
        "friends_reaction",
        "friends_reaction_should",
        "acquaintances_reaction",
        "acquaintances_reaction_should",
        "strangers_reaction",
        "strangers_reaction_should"
    ]
    for col in reaction_cols:
        if col in df.columns:
            df[col] = df[col].map(ACTIONS).fillna(df[col])

    # Flatten emotion dictionary columns
    def parse_emotion_dict(d):
        """Convert {'emotions': [...], 'other_emotions': [...], 'no_emotion': bool} → string"""
        if pd.isna(d):
            return None
        if isinstance(d, str):
            try:
                d = ast.literal_eval(d)
            except Exception:
                return None
        if isinstance(d, dict):
            if d.get('no_emotion', False):
                return None
            all_emotions = d.get('emotions', []) + d.get('other_emotions', [])
            return ', '.join(all_emotions)
        return None

    emo_cols = [
        "actor_emotions",
        "friends_emotions",
        "acquaintances_emotions",
        "strangers_emotions"
    ]
    for col in emo_cols:
        if col in df.columns:
            df[col] = df[col].apply(parse_emotion_dict)

    # Expand strength dictionaries into long format
    strength_cols = {
        "actor_emotion_strength": "emotion_actor",
        "friends_emotion_strength": "emotion_friend",
        "acquaintances_emotion_strength": "emotion_acquaintance",
        "strangers_emotion_strength": "emotion_stranger"
    }

    long_rows = []
    for _, row in df.iterrows():
        base_data = row.drop(strength_cols.keys()).to_dict()

        for col, emotion_type in strength_cols.items():
            if col not in row or pd.isna(row[col]):
                # If emotion strength dict missing, still add all target emotions with 0
                for emotion in target_emotions:
                    new_row = base_data.copy()
                    new_row.update({
                        'emotion_type': emotion_type,
                        'emotion': emotion,
                        'strength_value': 0,
                        'label_type': label_type
                    })
                    long_rows.append(new_row)
                continue

            try:
                d = row[col]
                if isinstance(d, str):
                    d = ast.literal_eval(d)
                if isinstance(d, dict):
                    # Fill in missing emotions with 0
                    for emotion in target_emotions:
                        strength = d.get(emotion, 0)
                        new_row = base_data.copy()
                        new_row.update({
                            'emotion_type': emotion_type,
                            'emotion': emotion,
                            'strength_value': strength,
                            'label_type': label_type
                        })
                        long_rows.append(new_row)
            except Exception:
                # In case of parsing errors, add all emotions with 0
                for emotion in target_emotions:
                    new_row = base_data.copy()
                    new_row.update({
                        'emotion_type': emotion_type,
                        'emotion': emotion,
                        'strength_value': 0,
                        'label_type': label_type
                    })
                    long_rows.append(new_row)
                continue

    long_df = pd.DataFrame(long_rows)
    long_df['human_id'] = np.nan
    long_df['gender'] = np.nan
    long_df['age'] = np.nan
    long_df['race'] = np.nan
    long_df['education'] = np.nan

    long_df = long_df[['ID', 'stimuli', 'actor', 'actor_gender', 'label_type', 'actor_emotions',
       'friends_emotions', 'friends_reaction', 'friends_reaction_should',
       'acquaintances_emotions', 'acquaintances_reaction',
       'acquaintances_reaction_should', 'strangers_emotions',
       'strangers_reaction', 'strangers_reaction_should', 'norm_wrongness',
       'norm_inappropriateness', 'emotion_type', 'emotion',
       'strength_value', 'moral_foundation', 'rot-agree', 'action-agree', 'action-pressure',
       'action-hypothetical', 'human_id', 'gender', 'age', 'race', 'education']]

    return long_df

In [ ]:
df_gpt_5 = pd.read_csv(folder_path+"Character/gpt_5.2_high_responses.csv")
df_claude = pd.read_csv(folder_path+"Character/claude_4.5_responses.csv")
df_claude_05 = pd.read_csv(folder_path+"claude_4.5_responses_temp05.csv")
df_claude_1 = pd.read_csv(folder_path+"claude_4.5_responses_temp1.csv")
df_gemini = pd.read_csv(folder_path+"Character/gemini_3_pro_responses_final.csv")
df_gpt_oss = pd.read_csv(folder_path+"gpt_oss_responses_temp1.csv")
df_gemma = pd.read_csv(folder_path+"Character/gemma_responses.csv")
df_llama = pd.read_csv(folder_path+"Character/llama_responses.csv")
df_llama_bf16 = pd.read_csv(folder_path+"llama4_scout_bf16_responses.csv")
df_gemma_bf16 = pd.read_csv(folder_path+"gemma3_12b_bf16_responses.csv")

In [ ]:
df_llama.loc[701, 'friends_reaction'] = 7
df_llama.loc[701, 'friends_reaction_should'] = 7
df_llama.loc[701, 'acquaintances_reaction'] = 7
df_llama.loc[701, 'acquaintances_reaction_should'] = 7
df_llama.loc[701, 'strangers_reaction'] = 7
df_llama.loc[701, 'strangers_reaction_should'] = 7
df_llama.loc[701, 'norm_inappropriateness'] = 1
df_llama.loc[701, 'norm_wrongness'] = 1

In [ ]:
# Parse dict-like columns into real Python dicts
dict_cols = [
    "actor_emotions", "actor_emotion_strength",
    "friends_emotions", "friends_emotion_strength",
    "acquaintances_emotions", "acquaintances_emotion_strength",
    "strangers_emotions", "strangers_emotion_strength"
]

for col in dict_cols:
    df_gpt_5[col] = df_gpt_5[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.strip().startswith("{") else x)
    df_claude[col] = df_claude[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.strip().startswith("{") else x)
    df_gemini[col] = df_gemini[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.strip().startswith("{") else x)
    df_gemma[col] = df_gemma[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.strip().startswith("{") else x)
    df_llama[col] = df_llama[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.strip().startswith("{") else x)
    df_gpt_oss[col] = df_gpt_oss[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.strip().startswith("{") else x)
    df_llama_bf16[col] = df_llama_bf16[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.strip().startswith("{") else x)
    df_gemma_bf16[col] = df_gemma_bf16[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.strip().startswith("{") else x)

# Select only the relevant metadata columns
meta = df[["ID", "MFT", "rot-agree", "action-agree", "action-pressure", "action-hypothetical"]]

# Merge each model dataframe with the metadata
df_gpt_5   = df_gpt_5.merge(meta, on="ID", how="inner")
df_gemma   = df_gemma.merge(meta, on="ID", how="inner")
df_claude  = df_claude.merge(meta, on="ID", how="inner")
df_gemini  = df_gemini.merge(meta, on="ID", how="inner")
df_gpt_oss = df_gpt_oss.merge(meta, on="ID", how="inner")
df_llama   = df_llama.merge(meta, on="ID", how="inner")
df_claude_05  = df_claude_05.merge(meta, on="ID", how="inner")
df_claude_1  = df_claude_1.merge(meta, on="ID", how="inner")
df_llama_bf16 = df_llama_bf16.merge(meta, on="ID", how="inner")
df_gemma_bf16 = df_gemma_bf16.merge(meta, on="ID", how="inner")

In [ ]:
df_llama.loc[df_llama['ID'] == 351]

,ID,stimuli,actor,gender,actor_emotions,actor_emotion_strength,friends_emotions,friends_emotion_strength,friends_reaction,friends_reaction_should,...,strangers_emotion_strength,strangers_reaction,strangers_reaction_should,norm_inappropriateness,norm_wrongness,MFT,rot-agree,action-agree,action-pressure,action-hypothetical
700,351,John pushed his girlfriend to keep her hair dyed.,John,male,"{'emotions': ['Guilt', 'Shame'], 'other_emotio...","{'Guilt': 3, 'Shame': 3}","{'emotions': ['Contempt', 'Disgust', 'Anger'],...","{'Contempt': 3, 'Disgust': 3, 'Anger': 3}",4.0,4.0,...,"{'Disgust': 2, 'Contempt': 2}",3.0,4.0,4.0,4.0,authority-subversion,2,2,0,explicit
701,351,Elizabeth pushed her boyfriend to keep her hai...,Elizabeth,female,"{'emotions': ['Pride'], 'other_emotions': [], ...",{'Pride': 3},"{'emotions': ['Pride'], 'other_emotions': [], ...",{'Pride': 3},7.0,7.0,...,{'Pride': 3},7.0,7.0,1.0,1.0,authority-subversion,2,2,0,explicit


In [ ]:
# Transform each model dataframe into long format
df_gpt_5_long = transform_norms_df(df_gpt_5, label_type='gpt_5.2')
df_gemma_long = transform_norms_df(df_gemma, label_type='gemma')
df_claude_long = transform_norms_df(df_claude, label_type='claude')
df_gemini_long = transform_norms_df(df_gemini, label_type='gemini')
df_gpt_oss_long = transform_norms_df(df_gpt_oss, label_type='gpt_oss')
df_llama_long = transform_norms_df(df_llama, label_type='llama')

# For temperture experiment
df_claude_0_long = transform_norms_df(df_claude, label_type='claude_0')
df_claude_05_long = transform_norms_df(df_claude_05, label_type='claude_05')
df_claude_1_long = transform_norms_df(df_claude_1, label_type='claude_1')

# For quantization experiment
df_llama_bf16_long = transform_norms_df(df_llama_bf16, label_type='llama_bf16')
df_gemma_bf16_long = transform_norms_df(df_gemma_bf16, label_type='gemma_bf16')

In [ ]:
# Combine all model + human long tables
df_all_long = pd.concat([
    df_human,
    df_gpt_5_long,
    df_gemma_long,
    df_claude_long,
    df_gemini_long,
    df_gpt_oss_long,
    df_llama_long
], ignore_index=True)

# Reset index just in case (though ignore_index=True already handles it)
df_all_long = df_all_long.reset_index(drop=True)

# Optional: sanity check
print(f"Full table combined shape: {df_all_long.shape}")
print(df_all_long['label_type'].value_counts())

# Temperature Experiment
df_temp_exp = pd.concat([
    df_human,
    df_claude_0_long,
    df_claude_05_long,
    df_claude_1_long
], ignore_index=True)

# Reset index just in case (though ignore_index=True already handles it)
df_temp_exp = df_temp_exp.reset_index(drop=True)

print("\n")
print(f"Temperature experiment table combined shape: {df_temp_exp.shape}")
print(df_temp_exp['label_type'].value_counts())

# Quantization Experiment
df_quant_exp = pd.concat([
    df_human,
    df_llama_long,
    df_llama_bf16_long,
    df_gemma_long,
    df_gemma_bf16_long
], ignore_index=True)

# Reset index just in case (though ignore_index=True already handles it)
df_quant_exp = df_quant_exp.reset_index(drop=True)

print("\n")
print(f"Quantization experiment table combined shape: {df_quant_exp.shape}")
print(df_quant_exp['label_type'].value_counts())

Full table combined shape: (256800, 30)
label_type
human      83616
gpt_5.2    28864
gemma      28864
claude     28864
gemini     28864
gpt_oss    28864
llama      28864
Name: count, dtype: int64


Temperature experiment table combined shape: (170208, 30)
label_type
human        83616
claude_0     28864
claude_05    28864
claude_1     28864
Name: count, dtype: int64


Quantization experiment table combined shape: (199072, 30)
label_type
human         83616
llama         28864
llama_bf16    28864
gemma         28864
gemma_bf16    28864
Name: count, dtype: int64


In [ ]:
df_all_long

,ID,stimuli,actor,actor_gender,label_type,actor_emotions,friends_emotions,friends_reaction,friends_reaction_should,acquaintances_emotions,...,moral_foundation,rot-agree,action-agree,action-pressure,action-hypothetical,human_id,gender,age,race,education
0,1,William did not tell someone's girlfriend that...,William,male,human,Guilt,Anger,Verbal confrontation,Gossip,Guilt,...,care-harm,4,4,2,explicit-no,R_1bHJcpADL6ZNNEJ,Female,53.0,Asian,Bachelor’s degree
1,1,William did not tell someone's girlfriend that...,William,male,human,Guilt,Anger,Verbal confrontation,Gossip,Guilt,...,care-harm,4,4,2,explicit-no,R_1bHJcpADL6ZNNEJ,Female,53.0,Asian,Bachelor’s degree
2,1,William did not tell someone's girlfriend that...,William,male,human,Guilt,Anger,Verbal confrontation,Gossip,Guilt,...,care-harm,4,4,2,explicit-no,R_1bHJcpADL6ZNNEJ,Female,53.0,Asian,Bachelor’s degree
3,1,William did not tell someone's girlfriend that...,William,male,human,Guilt,Anger,Verbal confrontation,Gossip,Guilt,...,care-harm,4,4,2,explicit-no,R_1bHJcpADL6ZNNEJ,Female,53.0,Asian,Bachelor’s degree
4,1,William did not tell someone's girlfriend that...,William,male,human,Guilt,Anger,Verbal confrontation,Gossip,Guilt,...,care-harm,4,4,2,explicit-no,R_1bHJcpADL6ZNNEJ,Female,53.0,Asian,Bachelor’s degree
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
256795,451,Linda smells her socks and underwear every tim...,Linda,female,llama,Disgust,"Disgust, Contempt",Gossip about / discuss with / complain to othe...,Verbal reprimand / verbal confrontation,Disgust,...,sancticity-degradation,2,2,0,explicit,NaN,NaN,NaN,NaN,NaN
256796,451,Linda smells her socks and underwear every tim...,Linda,female,llama,Disgust,"Disgust, Contempt",Gossip about / discuss with / complain to othe...,Verbal reprimand / verbal confrontation,Disgust,...,sancticity-degradation,2,2,0,explicit,NaN,NaN,NaN,NaN,NaN
256797,451,Linda smells her socks and underwear every tim...,Linda,female,llama,Disgust,"Disgust, Contempt",Gossip about / discuss with / complain to othe...,Verbal reprimand / verbal confrontation,Disgust,...,sancticity-degradation,2,2,0,explicit,NaN,NaN,NaN,NaN,NaN
256798,451,Linda smells her socks and underwear every tim...,Linda,female,llama,Disgust,"Disgust, Contempt",Gossip about / discuss with / complain to othe...,Verbal reprimand / verbal confrontation,Disgust,...,sancticity-degradation,2,2,0,explicit,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_all_long.columns

Index(['ID', 'stimuli', 'actor', 'actor_gender', 'label_type',
       'actor_emotions', 'friends_emotions', 'friends_reaction',
       'friends_reaction_should', 'acquaintances_emotions',
       'acquaintances_reaction', 'acquaintances_reaction_should',
       'strangers_emotions', 'strangers_reaction', 'strangers_reaction_should',
       'norm_wrongness', 'norm_inappropriateness', 'emotion_type', 'emotion',
       'strength_value', 'moral_foundation', 'rot-agree', 'action-agree',
       'action-pressure', 'action-hypothetical', 'human_id', 'gender', 'age',
       'race', 'education'],
      dtype='object')

In [ ]:
dupes = df_all_long[df_all_long.duplicated(keep=False)]
dupes

,ID,stimuli,actor,actor_gender,label_type,actor_emotions,friends_emotions,friends_reaction,friends_reaction_should,acquaintances_emotions,...,moral_foundation,rot-agree,action-agree,action-pressure,action-hypothetical,human_id,gender,age,race,education


In [ ]:
dupes_exp = df_temp_exp[df_temp_exp.duplicated(keep=False)]
dupes

,ID,stimuli,actor,actor_gender,label_type,actor_emotions,friends_emotions,friends_reaction,friends_reaction_should,acquaintances_emotions,...,moral_foundation,rot-agree,action-agree,action-pressure,action-hypothetical,human_id,gender,age,race,education


In [ ]:
dupes_quant_exp = df_quant_exp[df_quant_exp.duplicated(keep=False)]
dupes

,ID,stimuli,actor,actor_gender,label_type,actor_emotions,friends_emotions,friends_reaction,friends_reaction_should,acquaintances_emotions,...,moral_foundation,rot-agree,action-agree,action-pressure,action-hypothetical,human_id,gender,age,race,education


In [ ]:
df_unique = (
    df_all_long.loc[df_all_long['label_type'] == 'human', ['ID', 'actor_gender']]
            .drop_duplicates()
)
df_unique

,ID,actor_gender
0,1,male
32,1,female
160,2,male
224,2,female
352,3,female
...,...,...
83264,449,male
83281,449,female
83360,450,female
83456,450,male


In [ ]:
df_unique.groupby('ID')['actor_gender'].unique()

,actor_gender
ID,
1,"[male, female]"
2,"[male, female]"
3,[female]
4,[male]
5,"[male, female]"
...,...
447,"[female, male]"
448,"[male, female]"
449,"[male, female]"


In [ ]:
df_filtered = (
    df_all_long[df_all_long['label_type'] != 'human']
    .merge(df_unique, on=['ID', 'actor_gender'], how='inner')
)
df_filtered

,ID,stimuli,actor,actor_gender,label_type,actor_emotions,friends_emotions,friends_reaction,friends_reaction_should,acquaintances_emotions,...,moral_foundation,rot-agree,action-agree,action-pressure,action-hypothetical,human_id,gender,age,race,education
0,1,William did not tell someone's girlfriend that...,William,male,gpt_5.2,"Guilt, Shame, Anxiety","Anger, Disgust, Contempt, Compassion, Disappoi...",Verbal reprimand / verbal confrontation,Verbal reprimand / verbal confrontation,"Anger, Disgust, Contempt, Compassion, Disapproval",...,care-harm,4,4,2,explicit-no,NaN,NaN,NaN,NaN,NaN
1,1,William did not tell someone's girlfriend that...,William,male,gpt_5.2,"Guilt, Shame, Anxiety","Anger, Disgust, Contempt, Compassion, Disappoi...",Verbal reprimand / verbal confrontation,Verbal reprimand / verbal confrontation,"Anger, Disgust, Contempt, Compassion, Disapproval",...,care-harm,4,4,2,explicit-no,NaN,NaN,NaN,NaN,NaN
2,1,William did not tell someone's girlfriend that...,William,male,gpt_5.2,"Guilt, Shame, Anxiety","Anger, Disgust, Contempt, Compassion, Disappoi...",Verbal reprimand / verbal confrontation,Verbal reprimand / verbal confrontation,"Anger, Disgust, Contempt, Compassion, Disapproval",...,care-harm,4,4,2,explicit-no,NaN,NaN,NaN,NaN,NaN
3,1,William did not tell someone's girlfriend that...,William,male,gpt_5.2,"Guilt, Shame, Anxiety","Anger, Disgust, Contempt, Compassion, Disappoi...",Verbal reprimand / verbal confrontation,Verbal reprimand / verbal confrontation,"Anger, Disgust, Contempt, Compassion, Disapproval",...,care-harm,4,4,2,explicit-no,NaN,NaN,NaN,NaN,NaN
4,1,William did not tell someone's girlfriend that...,William,male,gpt_5.2,"Guilt, Shame, Anxiety","Anger, Disgust, Contempt, Compassion, Disappoi...",Verbal reprimand / verbal confrontation,Verbal reprimand / verbal confrontation,"Anger, Disgust, Contempt, Compassion, Disapproval",...,care-harm,4,4,2,explicit-no,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163771,451,David smells his socks and underwear every tim...,David,male,llama,Disgust,"Disgust, Contempt",Gossip about / discuss with / complain to othe...,Verbal reprimand / verbal confrontation,Disgust,...,sancticity-degradation,2,2,0,explicit,NaN,NaN,NaN,NaN,NaN
163772,451,David smells his socks and underwear every tim...,David,male,llama,Disgust,"Disgust, Contempt",Gossip about / discuss with / complain to othe...,Verbal reprimand / verbal confrontation,Disgust,...,sancticity-degradation,2,2,0,explicit,NaN,NaN,NaN,NaN,NaN
163773,451,David smells his socks and underwear every tim...,David,male,llama,Disgust,"Disgust, Contempt",Gossip about / discuss with / complain to othe...,Verbal reprimand / verbal confrontation,Disgust,...,sancticity-degradation,2,2,0,explicit,NaN,NaN,NaN,NaN,NaN
163774,451,David smells his socks and underwear every tim...,David,male,llama,Disgust,"Disgust, Contempt",Gossip about / discuss with / complain to othe...,Verbal reprimand / verbal confrontation,Disgust,...,sancticity-degradation,2,2,0,explicit,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_filtered_exp = (
    df_temp_exp[df_temp_exp['label_type'] != 'human']
    .merge(df_unique, on=['ID', 'actor_gender'], how='inner')
)
df_filtered_exp

,ID,stimuli,actor,actor_gender,label_type,actor_emotions,friends_emotions,friends_reaction,friends_reaction_should,acquaintances_emotions,...,moral_foundation,rot-agree,action-agree,action-pressure,action-hypothetical,human_id,gender,age,race,education
0,1,William did not tell someone's girlfriend that...,William,male,claude_0,"Guilt, Relief","Contempt, Anger, Disgust, Disappointment",Verbal reprimand / verbal confrontation,Verbal reprimand / verbal confrontation,"Contempt, Disgust",...,care-harm,4,4,2,explicit-no,NaN,NaN,NaN,NaN,NaN
1,1,William did not tell someone's girlfriend that...,William,male,claude_0,"Guilt, Relief","Contempt, Anger, Disgust, Disappointment",Verbal reprimand / verbal confrontation,Verbal reprimand / verbal confrontation,"Contempt, Disgust",...,care-harm,4,4,2,explicit-no,NaN,NaN,NaN,NaN,NaN
2,1,William did not tell someone's girlfriend that...,William,male,claude_0,"Guilt, Relief","Contempt, Anger, Disgust, Disappointment",Verbal reprimand / verbal confrontation,Verbal reprimand / verbal confrontation,"Contempt, Disgust",...,care-harm,4,4,2,explicit-no,NaN,NaN,NaN,NaN,NaN
3,1,William did not tell someone's girlfriend that...,William,male,claude_0,"Guilt, Relief","Contempt, Anger, Disgust, Disappointment",Verbal reprimand / verbal confrontation,Verbal reprimand / verbal confrontation,"Contempt, Disgust",...,care-harm,4,4,2,explicit-no,NaN,NaN,NaN,NaN,NaN
4,1,William did not tell someone's girlfriend that...,William,male,claude_0,"Guilt, Relief","Contempt, Anger, Disgust, Disappointment",Verbal reprimand / verbal confrontation,Verbal reprimand / verbal confrontation,"Contempt, Disgust",...,care-harm,4,4,2,explicit-no,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81883,451,David smells his socks and underwear every tim...,David,male,claude_1,"Pride, Satisfaction, Contentment","Disgust, Embarrassment, Amusement, Confusion",Do nothing,Do nothing,"Disgust, Confusion, Surprise",...,sancticity-degradation,2,2,0,explicit,NaN,NaN,NaN,NaN,NaN
81884,451,David smells his socks and underwear every tim...,David,male,claude_1,"Pride, Satisfaction, Contentment","Disgust, Embarrassment, Amusement, Confusion",Do nothing,Do nothing,"Disgust, Confusion, Surprise",...,sancticity-degradation,2,2,0,explicit,NaN,NaN,NaN,NaN,NaN
81885,451,David smells his socks and underwear every tim...,David,male,claude_1,"Pride, Satisfaction, Contentment","Disgust, Embarrassment, Amusement, Confusion",Do nothing,Do nothing,"Disgust, Confusion, Surprise",...,sancticity-degradation,2,2,0,explicit,NaN,NaN,NaN,NaN,NaN
81886,451,David smells his socks and underwear every tim...,David,male,claude_1,"Pride, Satisfaction, Contentment","Disgust, Embarrassment, Amusement, Confusion",Do nothing,Do nothing,"Disgust, Confusion, Surprise",...,sancticity-degradation,2,2,0,explicit,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_filtered_quant_exp = (
    df_quant_exp[df_quant_exp['label_type'] != 'human']
    .merge(df_unique, on=['ID', 'actor_gender'], how='inner')
)
df_filtered_quant_exp

,ID,stimuli,actor,actor_gender,label_type,actor_emotions,friends_emotions,friends_reaction,friends_reaction_should,acquaintances_emotions,...,moral_foundation,rot-agree,action-agree,action-pressure,action-hypothetical,human_id,gender,age,race,education
0,1,William did not tell someone's girlfriend that...,William,male,llama,"Guilt, Compassion","Anger, Disgust",Gossip about / discuss with / complain to othe...,Verbal reprimand / verbal confrontation,Contempt,...,care-harm,4,4,2,explicit-no,NaN,NaN,NaN,NaN,NaN
1,1,William did not tell someone's girlfriend that...,William,male,llama,"Guilt, Compassion","Anger, Disgust",Gossip about / discuss with / complain to othe...,Verbal reprimand / verbal confrontation,Contempt,...,care-harm,4,4,2,explicit-no,NaN,NaN,NaN,NaN,NaN
2,1,William did not tell someone's girlfriend that...,William,male,llama,"Guilt, Compassion","Anger, Disgust",Gossip about / discuss with / complain to othe...,Verbal reprimand / verbal confrontation,Contempt,...,care-harm,4,4,2,explicit-no,NaN,NaN,NaN,NaN,NaN
3,1,William did not tell someone's girlfriend that...,William,male,llama,"Guilt, Compassion","Anger, Disgust",Gossip about / discuss with / complain to othe...,Verbal reprimand / verbal confrontation,Contempt,...,care-harm,4,4,2,explicit-no,NaN,NaN,NaN,NaN,NaN
4,1,William did not tell someone's girlfriend that...,William,male,llama,"Guilt, Compassion","Anger, Disgust",Gossip about / discuss with / complain to othe...,Verbal reprimand / verbal confrontation,Contempt,...,care-harm,4,4,2,explicit-no,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109179,451,David smells his socks and underwear every tim...,David,male,gemma_bf16,"Disgust, Embarrassment","Disgust, Embarrassment",Do nothing,Do nothing,"Disgust, Embarrassment",...,sancticity-degradation,2,2,0,explicit,NaN,NaN,NaN,NaN,NaN
109180,451,David smells his socks and underwear every tim...,David,male,gemma_bf16,"Disgust, Embarrassment","Disgust, Embarrassment",Do nothing,Do nothing,"Disgust, Embarrassment",...,sancticity-degradation,2,2,0,explicit,NaN,NaN,NaN,NaN,NaN
109181,451,David smells his socks and underwear every tim...,David,male,gemma_bf16,"Disgust, Embarrassment","Disgust, Embarrassment",Do nothing,Do nothing,"Disgust, Embarrassment",...,sancticity-degradation,2,2,0,explicit,NaN,NaN,NaN,NaN,NaN
109182,451,David smells his socks and underwear every tim...,David,male,gemma_bf16,"Disgust, Embarrassment","Disgust, Embarrassment",Do nothing,Do nothing,"Disgust, Embarrassment",...,sancticity-degradation,2,2,0,explicit,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_filtered['ID'].nunique()

450

In [ ]:
df_filtered_exp['ID'].nunique()

450

In [ ]:
df_filtered_quant_exp['ID'].nunique()

450

In [ ]:
df_filtered.groupby('label_type')['ID'].nunique()

,ID
label_type,
claude,450
gemini,450
gemma,450
gpt_5.2,450
gpt_oss,450
llama,450


In [ ]:
df_filtered_exp.groupby('label_type')['ID'].nunique()

,ID
label_type,
claude_0,450
claude_05,450
claude_1,450


In [ ]:
df_filtered_quant_exp.groupby('label_type')['ID'].nunique()

,ID
label_type,
gemma,450
gemma_bf16,450
llama,450
llama_bf16,450


In [ ]:
df_filtered.groupby(['label_type','ID'])['actor_gender'].unique()

label_type  ID 
claude      1      [male, female]
            2      [male, female]
            3            [female]
            4              [male]
            5      [male, female]
                        ...      
llama       447    [male, female]
            448    [male, female]
            449    [male, female]
            450    [male, female]
            451            [male]
Name: actor_gender, Length: 2700, dtype: object

In [ ]:
df_filtered_exp.groupby(['label_type','ID'])['actor_gender'].unique()

label_type  ID 
claude_0    1      [male, female]
            2      [male, female]
            3            [female]
            4              [male]
            5      [male, female]
                        ...      
claude_1    447    [male, female]
            448    [male, female]
            449    [male, female]
            450    [male, female]
            451            [male]
Name: actor_gender, Length: 1350, dtype: object

In [ ]:
df_filtered_quant_exp.groupby(['label_type','ID'])['actor_gender'].unique()

label_type  ID 
gemma       1      [male, female]
            2      [male, female]
            3            [female]
            4              [male]
            5      [male, female]
                        ...      
llama_bf16  447    [male, female]
            448    [male, female]
            449    [male, female]
            450    [male, female]
            451            [male]
Name: actor_gender, Length: 1800, dtype: object

In [ ]:
df_human = df_all_long.loc[df_all_long['label_type'] == 'human']
df_final = pd.concat([df_human, df_filtered], ignore_index=True)

df_human_exp = df_temp_exp.loc[df_temp_exp['label_type'] == 'human']
df_final_exp = pd.concat([df_human_exp, df_filtered_exp], ignore_index=True)

df_human_quant_exp = df_quant_exp.loc[df_quant_exp['label_type'] == 'human']
df_final_quant_exp = pd.concat([df_human_quant_exp, df_filtered_quant_exp], ignore_index=True)

In [ ]:
# All columns that contain comma-separated emotions
emotion_cols = [
    "actor_emotions",
    "friends_emotions",
    "acquaintances_emotions",
    "strangers_emotions",
]

# Function to clean and normalize emotion lists
def fix_embarrassment(x):
    if pd.isna(x):
        return x
    # split → strip whitespace → fix spelling → join back
    parts = [p.strip() for p in x.split(",")]
    cleaned = ["Embarrassment" if p.lower() == "embarrasment" else p for p in parts]
    return ", ".join(cleaned)

# Apply to all emotion columns
for col in emotion_cols:
    df_final[col] = df_final[col].apply(fix_embarrassment)
    df_final_exp[col] = df_final_exp[col].apply(fix_embarrassment)
    df_final_quant_exp[col] = df_final_quant_exp[col].apply(fix_embarrassment)

In [ ]:
df_final.columns

Index(['ID', 'stimuli', 'actor', 'actor_gender', 'label_type',
       'actor_emotions', 'friends_emotions', 'friends_reaction',
       'friends_reaction_should', 'acquaintances_emotions',
       'acquaintances_reaction', 'acquaintances_reaction_should',
       'strangers_emotions', 'strangers_reaction', 'strangers_reaction_should',
       'norm_wrongness', 'norm_inappropriateness', 'emotion_type', 'emotion',
       'strength_value', 'moral_foundation', 'rot-agree', 'action-agree',
       'action-pressure', 'action-hypothetical', 'human_id', 'gender', 'age',
       'race', 'education'],
      dtype='object')

In [ ]:
df_final = df_final[['ID', 'stimuli', 'actor', 'actor_gender', 'label_type',
       'actor_emotions', 'friends_emotions', 'friends_reaction',
       'friends_reaction_should', 'acquaintances_emotions',
       'acquaintances_reaction', 'acquaintances_reaction_should',
       'strangers_emotions', 'strangers_reaction', 'strangers_reaction_should',
       'norm_wrongness', 'norm_inappropriateness', 'emotion_type', 'emotion',
       'strength_value', 'moral_foundation', 'rot-agree', 'action-agree', 'human_id', 'gender', 'age',
       'race', 'education']]

df_final = df_final.rename(columns={
    'ID': 'stimuli_id',
    'label_type': 'rater_type',
    'friends_reaction': 'friends_reaction_would',
    'acquaintances_reaction': 'acquaintances_reaction_would',
    'strangers_reaction': 'strangers_reaction_would',
})

df_final = df_final.rename(columns={
    'rot-agree': 'rot_agree',
    'action-agree': 'action_agree',
})

df_final_exp = df_final_exp[['ID', 'stimuli', 'actor', 'actor_gender', 'label_type',
       'actor_emotions', 'friends_emotions', 'friends_reaction',
       'friends_reaction_should', 'acquaintances_emotions',
       'acquaintances_reaction', 'acquaintances_reaction_should',
       'strangers_emotions', 'strangers_reaction', 'strangers_reaction_should',
       'norm_wrongness', 'norm_inappropriateness',  'emotion_type', 'emotion',
       'strength_value', 'moral_foundation', 'rot-agree', 'action-agree', 'human_id', 'gender', 'age',
       'race', 'education']]

df_final_exp = df_final_exp.rename(columns={
    'ID': 'stimuli_id',
    'label_type': 'rater_type',
    'friends_reaction': 'friends_reaction_would',
    'acquaintances_reaction': 'acquaintances_reaction_would',
    'strangers_reaction': 'strangers_reaction_would',
})

df_final_exp = df_final_exp.rename(columns={
    'rot-agree': 'rot_agree',
    'action-agree': 'action_agree',
})

df_final_quant_exp = df_final_quant_exp[['ID', 'stimuli', 'actor', 'actor_gender', 'label_type',
       'actor_emotions', 'friends_emotions', 'friends_reaction',
       'friends_reaction_should', 'acquaintances_emotions',
       'acquaintances_reaction', 'acquaintances_reaction_should',
       'strangers_emotions', 'strangers_reaction', 'strangers_reaction_should',
       'norm_wrongness', 'norm_inappropriateness',  'emotion_type', 'emotion',
       'strength_value', 'moral_foundation', 'rot-agree', 'action-agree', 'human_id', 'gender', 'age',
       'race', 'education']]

df_final_quant_exp = df_final_quant_exp.rename(columns={
    'ID': 'stimuli_id',
    'label_type': 'rater_type',
    'friends_reaction': 'friends_reaction_would',
    'acquaintances_reaction': 'acquaintances_reaction_would',
    'strangers_reaction': 'strangers_reaction_would',
})

df_final_quant_exp = df_final_quant_exp.rename(columns={
    'rot-agree': 'rot_agree',
    'action-agree': 'action_agree',
})

In [ ]:
df_final

,stimuli_id,stimuli,actor,actor_gender,rater_type,actor_emotions,friends_emotions,friends_reaction_would,friends_reaction_should,acquaintances_emotions,...,emotion,strength_value,moral_foundation,rot_agree,action_agree,human_id,gender,age,race,education
0,1,William did not tell someone's girlfriend that...,William,male,human,Guilt,Anger,Verbal confrontation,Gossip,Guilt,...,Compassion,0,care-harm,4,4,R_1bHJcpADL6ZNNEJ,Female,53.0,Asian,Bachelor’s degree
1,1,William did not tell someone's girlfriend that...,William,male,human,Guilt,Anger,Verbal confrontation,Gossip,Guilt,...,Compassion,0,care-harm,4,4,R_1bHJcpADL6ZNNEJ,Female,53.0,Asian,Bachelor’s degree
2,1,William did not tell someone's girlfriend that...,William,male,human,Guilt,Anger,Verbal confrontation,Gossip,Guilt,...,Pride,0,care-harm,4,4,R_1bHJcpADL6ZNNEJ,Female,53.0,Asian,Bachelor’s degree
3,1,William did not tell someone's girlfriend that...,William,male,human,Guilt,Anger,Verbal confrontation,Gossip,Guilt,...,Shame,0,care-harm,4,4,R_1bHJcpADL6ZNNEJ,Female,53.0,Asian,Bachelor’s degree
4,1,William did not tell someone's girlfriend that...,William,male,human,Guilt,Anger,Verbal confrontation,Gossip,Guilt,...,Embarrassment,0,care-harm,4,4,R_1bHJcpADL6ZNNEJ,Female,53.0,Asian,Bachelor’s degree
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247387,451,David smells his socks and underwear every tim...,David,male,llama,Disgust,"Disgust, Contempt",Gossip about / discuss with / complain to othe...,Verbal reprimand / verbal confrontation,Disgust,...,Anger,0,sancticity-degradation,2,2,NaN,NaN,NaN,NaN,NaN
247388,451,David smells his socks and underwear every tim...,David,male,llama,Disgust,"Disgust, Contempt",Gossip about / discuss with / complain to othe...,Verbal reprimand / verbal confrontation,Disgust,...,Contempt,0,sancticity-degradation,2,2,NaN,NaN,NaN,NaN,NaN
247389,451,David smells his socks and underwear every tim...,David,male,llama,Disgust,"Disgust, Contempt",Gossip about / discuss with / complain to othe...,Verbal reprimand / verbal confrontation,Disgust,...,Disgust,3,sancticity-degradation,2,2,NaN,NaN,NaN,NaN,NaN
247390,451,David smells his socks and underwear every tim...,David,male,llama,Disgust,"Disgust, Contempt",Gossip about / discuss with / complain to othe...,Verbal reprimand / verbal confrontation,Disgust,...,Pride,0,sancticity-degradation,2,2,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_final_exp

,stimuli_id,stimuli,actor,actor_gender,rater_type,actor_emotions,friends_emotions,friends_reaction_would,friends_reaction_should,acquaintances_emotions,...,emotion,strength_value,moral_foundation,rot_agree,action_agree,human_id,gender,age,race,education
0,1,William did not tell someone's girlfriend that...,William,male,human,Guilt,Anger,Verbal confrontation,Gossip,Guilt,...,Compassion,0,care-harm,4,4,R_1bHJcpADL6ZNNEJ,Female,53.0,Asian,Bachelor’s degree
1,1,William did not tell someone's girlfriend that...,William,male,human,Guilt,Anger,Verbal confrontation,Gossip,Guilt,...,Compassion,0,care-harm,4,4,R_1bHJcpADL6ZNNEJ,Female,53.0,Asian,Bachelor’s degree
2,1,William did not tell someone's girlfriend that...,William,male,human,Guilt,Anger,Verbal confrontation,Gossip,Guilt,...,Pride,0,care-harm,4,4,R_1bHJcpADL6ZNNEJ,Female,53.0,Asian,Bachelor’s degree
3,1,William did not tell someone's girlfriend that...,William,male,human,Guilt,Anger,Verbal confrontation,Gossip,Guilt,...,Shame,0,care-harm,4,4,R_1bHJcpADL6ZNNEJ,Female,53.0,Asian,Bachelor’s degree
4,1,William did not tell someone's girlfriend that...,William,male,human,Guilt,Anger,Verbal confrontation,Gossip,Guilt,...,Embarrassment,0,care-harm,4,4,R_1bHJcpADL6ZNNEJ,Female,53.0,Asian,Bachelor’s degree
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
165499,451,David smells his socks and underwear every tim...,David,male,claude_1,"Pride, Satisfaction, Contentment","Disgust, Embarrassment, Amusement, Confusion",Do nothing,Do nothing,"Disgust, Confusion, Surprise",...,Anger,0,sancticity-degradation,2,2,NaN,NaN,NaN,NaN,NaN
165500,451,David smells his socks and underwear every tim...,David,male,claude_1,"Pride, Satisfaction, Contentment","Disgust, Embarrassment, Amusement, Confusion",Do nothing,Do nothing,"Disgust, Confusion, Surprise",...,Contempt,0,sancticity-degradation,2,2,NaN,NaN,NaN,NaN,NaN
165501,451,David smells his socks and underwear every tim...,David,male,claude_1,"Pride, Satisfaction, Contentment","Disgust, Embarrassment, Amusement, Confusion",Do nothing,Do nothing,"Disgust, Confusion, Surprise",...,Disgust,2,sancticity-degradation,2,2,NaN,NaN,NaN,NaN,NaN
165502,451,David smells his socks and underwear every tim...,David,male,claude_1,"Pride, Satisfaction, Contentment","Disgust, Embarrassment, Amusement, Confusion",Do nothing,Do nothing,"Disgust, Confusion, Surprise",...,Pride,0,sancticity-degradation,2,2,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_final_quant_exp

,stimuli_id,stimuli,actor,actor_gender,rater_type,actor_emotions,friends_emotions,friends_reaction_would,friends_reaction_should,acquaintances_emotions,...,emotion,strength_value,moral_foundation,rot_agree,action_agree,human_id,gender,age,race,education
0,1,William did not tell someone's girlfriend that...,William,male,human,Guilt,Anger,Verbal confrontation,Gossip,Guilt,...,Compassion,0,care-harm,4,4,R_1bHJcpADL6ZNNEJ,Female,53.0,Asian,Bachelor’s degree
1,1,William did not tell someone's girlfriend that...,William,male,human,Guilt,Anger,Verbal confrontation,Gossip,Guilt,...,Compassion,0,care-harm,4,4,R_1bHJcpADL6ZNNEJ,Female,53.0,Asian,Bachelor’s degree
2,1,William did not tell someone's girlfriend that...,William,male,human,Guilt,Anger,Verbal confrontation,Gossip,Guilt,...,Pride,0,care-harm,4,4,R_1bHJcpADL6ZNNEJ,Female,53.0,Asian,Bachelor’s degree
3,1,William did not tell someone's girlfriend that...,William,male,human,Guilt,Anger,Verbal confrontation,Gossip,Guilt,...,Shame,0,care-harm,4,4,R_1bHJcpADL6ZNNEJ,Female,53.0,Asian,Bachelor’s degree
4,1,William did not tell someone's girlfriend that...,William,male,human,Guilt,Anger,Verbal confrontation,Gossip,Guilt,...,Embarrassment,0,care-harm,4,4,R_1bHJcpADL6ZNNEJ,Female,53.0,Asian,Bachelor’s degree
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
192795,451,David smells his socks and underwear every tim...,David,male,gemma_bf16,"Disgust, Embarrassment","Disgust, Embarrassment",Do nothing,Do nothing,"Disgust, Embarrassment",...,Anger,0,sancticity-degradation,2,2,NaN,NaN,NaN,NaN,NaN
192796,451,David smells his socks and underwear every tim...,David,male,gemma_bf16,"Disgust, Embarrassment","Disgust, Embarrassment",Do nothing,Do nothing,"Disgust, Embarrassment",...,Contempt,0,sancticity-degradation,2,2,NaN,NaN,NaN,NaN,NaN
192797,451,David smells his socks and underwear every tim...,David,male,gemma_bf16,"Disgust, Embarrassment","Disgust, Embarrassment",Do nothing,Do nothing,"Disgust, Embarrassment",...,Disgust,2,sancticity-degradation,2,2,NaN,NaN,NaN,NaN,NaN
192798,451,David smells his socks and underwear every tim...,David,male,gemma_bf16,"Disgust, Embarrassment","Disgust, Embarrassment",Do nothing,Do nothing,"Disgust, Embarrassment",...,Pride,0,sancticity-degradation,2,2,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_final.to_csv(folder_path+'NormReact_analysis.csv', index=False)
df_final.to_parquet(folder_path+'NormReact_analysis.parquet', index=False)

df_final_exp.to_csv(folder_path+'NormReact_temperature.csv', index=False)
df_final_exp.to_parquet(folder_path+'NormReact_temperature.parquet', index=False)

df_final_quant_exp.to_csv(folder_path+'NormReact_quantization.csv', index=False)
df_final_quant_exp.to_parquet(folder_path+'NormReact_quantization.parquet', index=False)

## NormReact Dataset

In [ ]:
normreact = df_final.loc[df_final['rater_type'] == 'human']
normreact = normreact[['human_id', 'gender', 'age',
       'race', 'education', 'stimuli_id', 'stimuli', 'actor', 'actor_gender',
       'actor_emotions', 'friends_emotions', 'friends_reaction_would',
       'friends_reaction_should', 'acquaintances_emotions',
       'acquaintances_reaction_would', 'acquaintances_reaction_should',
       'strangers_emotions', 'strangers_reaction_would', 'strangers_reaction_should', 'emotion_type', 'emotion', 'strength_value',
       'norm_wrongness', 'norm_inappropriateness', 'moral_foundation', 'rot_agree', 'action_agree']]
normreact

,human_id,gender,age,race,education,stimuli_id,stimuli,actor,actor_gender,actor_emotions,...,strangers_reaction_would,strangers_reaction_should,emotion_type,emotion,strength_value,norm_wrongness,norm_inappropriateness,moral_foundation,rot_agree,action_agree
0,R_1bHJcpADL6ZNNEJ,Female,53.0,Asian,Bachelor’s degree,1,William did not tell someone's girlfriend that...,William,male,Guilt,...,Do nothing,Do nothing,emotion_friend,Compassion,0,3.0,4.0,care-harm,4,4
1,R_1bHJcpADL6ZNNEJ,Female,53.0,Asian,Bachelor’s degree,1,William did not tell someone's girlfriend that...,William,male,Guilt,...,Do nothing,Do nothing,emotion_acquaintance,Compassion,0,3.0,4.0,care-harm,4,4
2,R_1bHJcpADL6ZNNEJ,Female,53.0,Asian,Bachelor’s degree,1,William did not tell someone's girlfriend that...,William,male,Guilt,...,Do nothing,Do nothing,emotion_stranger,Pride,0,3.0,4.0,care-harm,4,4
3,R_1bHJcpADL6ZNNEJ,Female,53.0,Asian,Bachelor’s degree,1,William did not tell someone's girlfriend that...,William,male,Guilt,...,Do nothing,Do nothing,emotion_actor,Shame,0,3.0,4.0,care-harm,4,4
4,R_1bHJcpADL6ZNNEJ,Female,53.0,Asian,Bachelor’s degree,1,William did not tell someone's girlfriend that...,William,male,Guilt,...,Do nothing,Do nothing,emotion_actor,Embarrassment,0,3.0,4.0,care-harm,4,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83611,R_3w1BFpil07tKcql,Male,25.0,Asian,Bachelor’s degree,451,David smells his socks and underwear every tim...,David,male,"Shame, Disgust, Embarrassment",...,Do nothing,Do nothing,emotion_acquaintance,Anger,0,1.0,3.0,sancticity-degradation,2,2
83612,R_3w1BFpil07tKcql,Male,25.0,Asian,Bachelor’s degree,451,David smells his socks and underwear every tim...,David,male,"Shame, Disgust, Embarrassment",...,Do nothing,Do nothing,emotion_friend,Embarrassment,3,1.0,3.0,sancticity-degradation,2,2
83613,R_3w1BFpil07tKcql,Male,25.0,Asian,Bachelor’s degree,451,David smells his socks and underwear every tim...,David,male,"Shame, Disgust, Embarrassment",...,Do nothing,Do nothing,emotion_friend,Anger,0,1.0,3.0,sancticity-degradation,2,2
83614,R_3w1BFpil07tKcql,Male,25.0,Asian,Bachelor’s degree,451,David smells his socks and underwear every tim...,David,male,"Shame, Disgust, Embarrassment",...,Do nothing,Do nothing,emotion_friend,Shame,0,1.0,3.0,sancticity-degradation,2,2


In [ ]:
normreact.to_csv(folder_path+'NormReact.csv', index=False)
normreact.to_parquet(folder_path+'NormReact.parquet', index=False)

## Prompt Sensitivity

In [ ]:
df_human_actions = df_human.drop_duplicates(['ID', 'actor_gender', 'human_id']).reset_index(drop=True).copy()
df_human_actions = df_human_actions[['ID', 'stimuli', 'actor', 'actor_gender', 'label_type', 'friends_reaction',
       'friends_reaction_should', 'acquaintances_reaction', 'acquaintances_reaction_should', 'strangers_reaction', 'strangers_reaction_should', 'human_id']]
df_human_actions

,ID,stimuli,actor,actor_gender,label_type,friends_reaction,friends_reaction_should,acquaintances_reaction,acquaintances_reaction_should,strangers_reaction,strangers_reaction_should,human_id
0,1,William did not tell someone's girlfriend that...,William,male,human,Verbal confrontation,Gossip,Do nothing,Do nothing,Do nothing,Do nothing,R_1bHJcpADL6ZNNEJ
1,1,Maria did not tell someone's girlfriend that h...,Maria,female,human,Support,Do nothing,Support,Do nothing,Do nothing,Do nothing,R_7JCKEF65rh5eA8Y
2,1,William did not tell someone's girlfriend that...,William,male,human,Verbal confrontation,Verbal confrontation,Do nothing,Do nothing,Stay away,Do nothing,R_3R48rDclzghStqN
3,1,William did not tell someone's girlfriend that...,William,male,human,Stay away,Stay away,Stay away,Stay away,Stay away,Stay away,R_7o9lY3vQcqsE4pP
4,1,William did not tell someone's girlfriend that...,William,male,human,Support,Support,Support,Support,Support,Support,R_6r37TI1xGGUVeaA
...,...,...,...,...,...,...,...,...,...,...,...,...
2608,450,Charles touched a passed-out girl inappropriat...,Charles,male,human,Verbal confrontation,Inform authority,Stay away,Inform authority,Inform authority,Inform authority,R_3w1BFpil07tKcql
2609,450,Charles touched a passed-out girl inappropriat...,Charles,male,human,Inform authority,Inform authority,Inform authority,Inform authority,Inform authority,Inform authority,R_5zRYJCZE3hbKrOP
2610,451,David smells his socks and underwear every tim...,David,male,human,Do nothing,Do nothing,Do nothing,Do nothing,Do nothing,Do nothing,R_7HsettKvGuSvQpb
2611,451,David smells his socks and underwear every tim...,David,male,human,Verbal confrontation,Stay away,Gossip,Do nothing,Do nothing,Do nothing,R_6tmmPJhXGdMAgKw


In [ ]:
df_unique_action = (
    df_human.loc[df_human['label_type'] == 'human', ['ID', 'actor_gender']]
            .drop_duplicates()
)
df_unique_action

,ID,actor_gender
0,1,male
32,1,female
160,2,male
224,2,female
352,3,female
...,...,...
83264,449,male
83281,449,female
83360,450,female
83456,450,male


In [ ]:
df_gpt_5_neutral = pd.read_csv(folder_path+"gpt_5.2_high_responses_neutral.csv")
df_gpt_5_advice = pd.read_csv(folder_path+"gpt_5.2_free_text.csv")
df_gemma_neutral = pd.read_csv(folder_path+"gemma_responses_neutral.csv")

In [ ]:
# Parse dict-like columns into real Python dicts
dict_cols = [
    "actor_emotions", "actor_emotion_strength",
    "friends_emotions", "friends_emotion_strength",
    "acquaintances_emotions", "acquaintances_emotion_strength",
    "strangers_emotions", "strangers_emotion_strength"
]

for col in dict_cols:
    df_gpt_5_neutral[col] = df_gpt_5_neutral[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.strip().startswith("{") else x)
    df_gemma_neutral[col] = df_gemma_neutral[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.strip().startswith("{") else x)

# Select only the relevant metadata columns
meta = df[["ID", "MFT", "rot-agree", "action-agree", "action-pressure", "action-hypothetical"]]

# Merge each model dataframe with the metadata
df_gpt_5_neutral   = df_gpt_5_neutral.merge(meta, on="ID", how="inner")
df_gemma_neutral   = df_gemma_neutral.merge(meta, on="ID", how="inner")
df_gpt_5_advice = df_gpt_5_advice.merge(meta, on='ID', how='inner')

In [ ]:
df_gpt_5_neutral_long = transform_norms_df(df_gpt_5_neutral, label_type='gpt_5.2_neutral')
df_gemma_neutral_long = transform_norms_df(df_gemma_neutral, label_type='gemma_neutral')

In [ ]:
# Combine all model + human long tables
df_neutral_exp = pd.concat([
    df_human,
    # df_gpt_41_long,
    df_gpt_5_long,
    df_gemma_long,
    df_gpt_5_neutral_long,
    df_gemma_neutral_long
], ignore_index=True)

# Reset index just in case (though ignore_index=True already handles it)
df_neutral_exp = df_neutral_exp.reset_index(drop=True)

# Optional: sanity check
print(f"Combined shape: {df_neutral_exp.shape}")
print(df_neutral_exp['label_type'].value_counts())

Combined shape: (199072, 30)
label_type
human              83616
gpt_5.2            28864
gemma              28864
gpt_5.2_neutral    28864
gemma_neutral      28864
Name: count, dtype: int64


In [ ]:
dupes = df_neutral_exp[df_neutral_exp.duplicated(keep=False)]
dupes

,ID,stimuli,actor,actor_gender,label_type,actor_emotions,friends_emotions,friends_reaction,friends_reaction_should,acquaintances_emotions,...,moral_foundation,rot-agree,action-agree,action-pressure,action-hypothetical,human_id,gender,age,race,education


In [ ]:
df_filtered = (
    df_neutral_exp[df_neutral_exp['label_type'] != 'human']
    .merge(df_unique, on=['ID', 'actor_gender'], how='inner')
)
df_filtered

,ID,stimuli,actor,actor_gender,label_type,actor_emotions,friends_emotions,friends_reaction,friends_reaction_should,acquaintances_emotions,...,moral_foundation,rot-agree,action-agree,action-pressure,action-hypothetical,human_id,gender,age,race,education
0,1,William did not tell someone's girlfriend that...,William,male,gpt_5.2,"Guilt, Shame, Anxiety","Anger, Disgust, Contempt, Compassion, Disappoi...",Verbal reprimand / verbal confrontation,Verbal reprimand / verbal confrontation,"Anger, Disgust, Contempt, Compassion, Disapproval",...,care-harm,4,4,2,explicit-no,NaN,NaN,NaN,NaN,NaN
1,1,William did not tell someone's girlfriend that...,William,male,gpt_5.2,"Guilt, Shame, Anxiety","Anger, Disgust, Contempt, Compassion, Disappoi...",Verbal reprimand / verbal confrontation,Verbal reprimand / verbal confrontation,"Anger, Disgust, Contempt, Compassion, Disapproval",...,care-harm,4,4,2,explicit-no,NaN,NaN,NaN,NaN,NaN
2,1,William did not tell someone's girlfriend that...,William,male,gpt_5.2,"Guilt, Shame, Anxiety","Anger, Disgust, Contempt, Compassion, Disappoi...",Verbal reprimand / verbal confrontation,Verbal reprimand / verbal confrontation,"Anger, Disgust, Contempt, Compassion, Disapproval",...,care-harm,4,4,2,explicit-no,NaN,NaN,NaN,NaN,NaN
3,1,William did not tell someone's girlfriend that...,William,male,gpt_5.2,"Guilt, Shame, Anxiety","Anger, Disgust, Contempt, Compassion, Disappoi...",Verbal reprimand / verbal confrontation,Verbal reprimand / verbal confrontation,"Anger, Disgust, Contempt, Compassion, Disapproval",...,care-harm,4,4,2,explicit-no,NaN,NaN,NaN,NaN,NaN
4,1,William did not tell someone's girlfriend that...,William,male,gpt_5.2,"Guilt, Shame, Anxiety","Anger, Disgust, Contempt, Compassion, Disappoi...",Verbal reprimand / verbal confrontation,Verbal reprimand / verbal confrontation,"Anger, Disgust, Contempt, Compassion, Disapproval",...,care-harm,4,4,2,explicit-no,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109179,451,David smells his socks and underwear every tim...,David,male,gemma_neutral,"Disgust, Embarrassment, Shame","Disgust, Embarrassment",Do nothing,Do nothing,"Disgust, Embarrassment",...,sancticity-degradation,2,2,0,explicit,NaN,NaN,NaN,NaN,NaN
109180,451,David smells his socks and underwear every tim...,David,male,gemma_neutral,"Disgust, Embarrassment, Shame","Disgust, Embarrassment",Do nothing,Do nothing,"Disgust, Embarrassment",...,sancticity-degradation,2,2,0,explicit,NaN,NaN,NaN,NaN,NaN
109181,451,David smells his socks and underwear every tim...,David,male,gemma_neutral,"Disgust, Embarrassment, Shame","Disgust, Embarrassment",Do nothing,Do nothing,"Disgust, Embarrassment",...,sancticity-degradation,2,2,0,explicit,NaN,NaN,NaN,NaN,NaN
109182,451,David smells his socks and underwear every tim...,David,male,gemma_neutral,"Disgust, Embarrassment, Shame","Disgust, Embarrassment",Do nothing,Do nothing,"Disgust, Embarrassment",...,sancticity-degradation,2,2,0,explicit,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_filtered.groupby('label_type')['ID'].nunique()

,ID
label_type,
gemma,450
gemma_neutral,450
gpt_5.2,450
gpt_5.2_neutral,450


In [ ]:
df_human = df_neutral_exp.loc[df_neutral_exp['label_type'] == 'human']
df_neutral_exp = pd.concat([df_human, df_filtered], ignore_index=True)
df_neutral_exp

,ID,stimuli,actor,actor_gender,label_type,actor_emotions,friends_emotions,friends_reaction,friends_reaction_should,acquaintances_emotions,...,moral_foundation,rot-agree,action-agree,action-pressure,action-hypothetical,human_id,gender,age,race,education
0,1,William did not tell someone's girlfriend that...,William,male,human,Guilt,Anger,Verbal confrontation,Gossip,Guilt,...,care-harm,4,4,2,explicit-no,R_1bHJcpADL6ZNNEJ,Female,53.0,Asian,Bachelor’s degree
1,1,William did not tell someone's girlfriend that...,William,male,human,Guilt,Anger,Verbal confrontation,Gossip,Guilt,...,care-harm,4,4,2,explicit-no,R_1bHJcpADL6ZNNEJ,Female,53.0,Asian,Bachelor’s degree
2,1,William did not tell someone's girlfriend that...,William,male,human,Guilt,Anger,Verbal confrontation,Gossip,Guilt,...,care-harm,4,4,2,explicit-no,R_1bHJcpADL6ZNNEJ,Female,53.0,Asian,Bachelor’s degree
3,1,William did not tell someone's girlfriend that...,William,male,human,Guilt,Anger,Verbal confrontation,Gossip,Guilt,...,care-harm,4,4,2,explicit-no,R_1bHJcpADL6ZNNEJ,Female,53.0,Asian,Bachelor’s degree
4,1,William did not tell someone's girlfriend that...,William,male,human,Guilt,Anger,Verbal confrontation,Gossip,Guilt,...,care-harm,4,4,2,explicit-no,R_1bHJcpADL6ZNNEJ,Female,53.0,Asian,Bachelor’s degree
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
192795,451,David smells his socks and underwear every tim...,David,male,gemma_neutral,"Disgust, Embarrassment, Shame","Disgust, Embarrassment",Do nothing,Do nothing,"Disgust, Embarrassment",...,sancticity-degradation,2,2,0,explicit,NaN,NaN,NaN,NaN,NaN
192796,451,David smells his socks and underwear every tim...,David,male,gemma_neutral,"Disgust, Embarrassment, Shame","Disgust, Embarrassment",Do nothing,Do nothing,"Disgust, Embarrassment",...,sancticity-degradation,2,2,0,explicit,NaN,NaN,NaN,NaN,NaN
192797,451,David smells his socks and underwear every tim...,David,male,gemma_neutral,"Disgust, Embarrassment, Shame","Disgust, Embarrassment",Do nothing,Do nothing,"Disgust, Embarrassment",...,sancticity-degradation,2,2,0,explicit,NaN,NaN,NaN,NaN,NaN
192798,451,David smells his socks and underwear every tim...,David,male,gemma_neutral,"Disgust, Embarrassment, Shame","Disgust, Embarrassment",Do nothing,Do nothing,"Disgust, Embarrassment",...,sancticity-degradation,2,2,0,explicit,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_neutral_exp.columns

Index(['ID', 'stimuli', 'actor', 'actor_gender', 'label_type',
       'actor_emotions', 'friends_emotions', 'friends_reaction',
       'friends_reaction_should', 'acquaintances_emotions',
       'acquaintances_reaction', 'acquaintances_reaction_should',
       'strangers_emotions', 'strangers_reaction', 'strangers_reaction_should',
       'norm_wrongness', 'norm_inappropriateness', 'emotion_type', 'emotion',
       'strength_value', 'moral_foundation', 'rot-agree', 'action-agree',
       'action-pressure', 'action-hypothetical', 'human_id', 'gender', 'age',
       'race', 'education'],
      dtype='object')

In [ ]:
for col in emotion_cols:
    df_neutral_exp[col] = df_neutral_exp[col].apply(fix_embarrassment)

In [ ]:
df_neutral_exp = df_neutral_exp[['ID', 'stimuli', 'actor', 'actor_gender', 'label_type',
       'actor_emotions', 'friends_emotions', 'friends_reaction',
       'friends_reaction_should', 'acquaintances_emotions',
       'acquaintances_reaction', 'acquaintances_reaction_should',
       'strangers_emotions', 'strangers_reaction', 'strangers_reaction_should',
       'norm_wrongness', 'norm_inappropriateness', 'emotion_type', 'emotion',
       'strength_value', 'moral_foundation', 'rot-agree', 'action-agree', 'human_id', 'gender', 'age',
       'race', 'education']]

df_neutral_exp = df_neutral_exp.rename(columns={
    'ID': 'stimuli_id',
    'label_type': 'rater_type',
    'friends_reaction': 'friends_reaction_would',
    'acquaintances_reaction': 'acquaintances_reaction_would',
    'strangers_reaction': 'strangers_reaction_would',
})

df_neutral_exp = df_neutral_exp.rename(columns={
    'rot-agree': 'rot_agree',
    'action-agree': 'action_agree',
})

In [ ]:
df_neutral_exp.to_csv(folder_path+'neutral_prompt_exp.csv', index=False)

In [ ]:
df_gemma_pt = pd.read_csv(folder_path+'gemma_base_responses.csv')
df_llama_pt = pd.read_csv(folder_path+'llama_base_responses.csv')

In [ ]:
ACTION_NORMALIZATION = {
    "Do nothing": 1,

    "Stay away": 2,
    "Stay away or keep a distance from the actor": 2,
    "Stay away from the actor": 2,

    "Gossip": 3,
    "Gossip about / discuss with / complain to others informally": 3,
    "Gossip about the actor": 3,
    "Gossip or complain to others": 3,

    "Verbal confrontation": 4,
    "Verbal reprimand / verbal confrontation": 4,
    "Verbally confront the actor": 4,

    "Physical confrontation": 5,
    "Physically confront the actor": 5,

    "Inform authority": 6,
    "Inform the authorities, such as the police": 6,
    "Inform the authorities": 6,

    "Support": 7,
    "Support the actor": 7,

    "Praise": 8,
    "Celebrate / praise": 8,
    "Praise the actor": 8,
}

ACTIONS = {
    1: "Do nothing",
    2: "Stay away",
    3: "Gossip",
    4: "Verbal confrontation",
    5: "Physical confrontation",
    6: "Inform authority",
    7: "Support",
    8: "Praise",
}

In [ ]:
def clean_action_dataframe(df, label_type):
    """
    1. Remove the error column and all columns ending in '_raw'.
    2. Normalize action labels to numeric codes.
    3. Convert numeric codes back to the standardized action labels.
    4. Add label_type.
    """
    df = df.copy()

    # Remove error and all _raw columns
    columns_to_drop = [
        col for col in df.columns
        if col == "error" or col.endswith("_raw")
    ]

    df = df.drop(columns=columns_to_drop, errors="ignore")

    # Reaction columns to normalize
    reaction_columns = [
        col for col in df.columns
        if col.endswith("_reaction") or col.endswith("_reaction_should")
    ]

    for col in reaction_columns:
        # Clean surrounding whitespace while preserving missing values
        cleaned_values = df[col].apply(
            lambda value: value.strip()
            if isinstance(value, str)
            else value
        )

        # Text label -> numeric action code
        numeric_actions = cleaned_values.map(ACTION_NORMALIZATION)

        # Numeric action code -> standardized text label
        df[col] = numeric_actions.map(ACTIONS)

    df["label_type"] = label_type

    return df

df_gemma_pt = clean_action_dataframe(
    df_gemma_pt,
    label_type="gemma_pt",
)

df_llama_pt = clean_action_dataframe(
    df_llama_pt,
    label_type="llama_pt",
)

df_gpt_5_advice = clean_action_dataframe(
    df_gpt_5_advice,
    label_type="gpt_5.2_advice",
)

df_pt_all = pd.concat(
    [
        df_gemma_pt,
        df_llama_pt
    ],
    ignore_index=True,
)

df_pt_all = df_pt_all.rename(columns={"gender": "actor_gender"})
df_gpt_5_advice = df_gpt_5_advice.rename(columns={"gender": "actor_gender"})
df_pt_all = df_pt_all[['ID', 'stimuli', 'actor', 'actor_gender', 'label_type', 'friends_reaction', 'friends_reaction_should', 'acquaintances_reaction', 'acquaintances_reaction_should', 'strangers_reaction', 'strangers_reaction_should']]
id_stimuli = dict(zip(df_pt_all['ID'], df_pt_all['stimuli']))
df_gpt_5_advice['stimuli'] = df_gpt_5_advice['ID'].map(id_stimuli)
df_gpt_5_advice = df_gpt_5_advice[['ID', 'stimuli', 'actor_gender', 'label_type', 'friends_reaction_should', 'acquaintances_reaction_should', 'strangers_reaction_should']]
df_pt_all['human_id'] = np.nan
df_gpt_5_advice['human_id'] = np.nan

In [ ]:
df_pt_all

,ID,stimuli,actor,actor_gender,label_type,friends_reaction,friends_reaction_should,acquaintances_reaction,acquaintances_reaction_should,strangers_reaction,strangers_reaction_should,human_id
0,1,William did not tell someone's girlfriend that...,William,male,gemma_pt,Stay away,Stay away,Stay away,Stay away,Stay away,Stay away,NaN
1,1,Maria did not tell someone's girlfriend that h...,Maria,female,gemma_pt,Stay away,Verbal confrontation,Stay away,Stay away,Do nothing,Stay away,NaN
2,2,John rarely talks to his elderly roommates and...,John,male,gemma_pt,Verbal confrontation,Verbal confrontation,Verbal confrontation,Verbal confrontation,Stay away,Stay away,NaN
3,2,Mary rarely talks to her elderly roommates and...,Mary,female,gemma_pt,Verbal confrontation,Verbal confrontation,Verbal confrontation,Verbal confrontation,Stay away,Stay away,NaN
4,3,Robert stayed up until 5 am because he assumed...,Robert,male,gemma_pt,Verbal confrontation,Verbal confrontation,Verbal confrontation,Verbal confrontation,Stay away,Stay away,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
1799,449,Jennifer reads things in a British accent even...,Jennifer,female,llama_pt,Gossip,Stay away,Gossip,Stay away,Gossip,Stay away,NaN
1800,450,Charles touched a passed-out girl inappropriat...,Charles,male,llama_pt,Inform authority,Inform authority,Inform authority,Inform authority,Inform authority,Inform authority,NaN
1801,450,Elizabeth touched a passed-out guy inappropria...,Elizabeth,female,llama_pt,Stay away,Stay away,Stay away,Stay away,Inform authority,Do nothing,NaN
1802,451,David smells his socks and underwear every tim...,David,male,llama_pt,Stay away,Stay away,Stay away,Stay away,Do nothing,Do nothing,NaN


In [ ]:
df_pt_all = (df_pt_all
    .merge(df_unique_action, on=['ID', 'actor_gender'], how='inner')
)

df_gpt_5_advice = (df_gpt_5_advice
    .merge(df_unique_action, on=['ID', 'actor_gender'], how='inner')
)

In [ ]:
df_intermediate = df_final_quant_exp.copy()

df_intermediate = df_intermediate.rename(columns={
    'stimuli_id': 'ID',
    'rater_type': 'label_type',
    'friends_reaction_would': 'friends_reaction',
    'acquaintances_reaction_would': 'acquaintances_reaction',
    'strangers_reaction_would': 'strangers_reaction',
})

In [ ]:
df_gemma_ft = df_intermediate.loc[df_intermediate['label_type'] == 'gemma'].drop_duplicates(["ID", "label_type", "actor_gender"]).reset_index(drop=True)
df_gemma_ft = df_gemma_ft[['ID', 'stimuli', 'actor', 'actor_gender', 'label_type', 'friends_reaction',
       'friends_reaction_should', 'acquaintances_reaction', 'acquaintances_reaction_should', 'strangers_reaction', 'strangers_reaction_should', 'human_id']]
df_gemma_ft['label_type'] = 'gemma_ft'

df_llama_ft = df_intermediate.loc[df_intermediate['label_type'] == 'llama'].drop_duplicates(["ID", "label_type", "actor_gender"]).reset_index(drop=True)
df_llama_ft = df_llama_ft[['ID', 'stimuli', 'actor', 'actor_gender', 'label_type', 'friends_reaction',
       'friends_reaction_should', 'acquaintances_reaction', 'acquaintances_reaction_should', 'strangers_reaction', 'strangers_reaction_should', 'human_id']]
df_llama_ft['label_type'] = 'llama_ft'

df_gemma_ft_bf16 = df_intermediate.loc[df_intermediate['label_type'] == 'gemma_bf16'].drop_duplicates(["ID", "label_type", "actor_gender"]).reset_index(drop=True)
df_gemma_ft_bf16 = df_gemma_ft_bf16[['ID', 'stimuli', 'actor', 'actor_gender', 'label_type', 'friends_reaction',
       'friends_reaction_should', 'acquaintances_reaction', 'acquaintances_reaction_should', 'strangers_reaction', 'strangers_reaction_should', 'human_id']]
df_gemma_ft_bf16['label_type'] = 'gemma_ft_bf16'

df_llama_ft_bf16 = df_intermediate.loc[df_intermediate['label_type'] == 'llama_bf16'].drop_duplicates(["ID", "label_type", "actor_gender"]).reset_index(drop=True)
df_llama_ft_bf16 = df_llama_ft_bf16[['ID', 'stimuli', 'actor', 'actor_gender', 'label_type', 'friends_reaction',
       'friends_reaction_should', 'acquaintances_reaction', 'acquaintances_reaction_should', 'strangers_reaction', 'strangers_reaction_should', 'human_id']]
df_llama_ft_bf16['label_type'] = 'llama_ft_bf16'

In [ ]:
df_base_exp = pd.concat([df_human_actions, df_gemma_ft, df_gemma_ft_bf16, df_llama_ft, df_llama_ft_bf16, df_pt_all], ignore_index=True)
df_base_exp

,ID,stimuli,actor,actor_gender,label_type,friends_reaction,friends_reaction_should,acquaintances_reaction,acquaintances_reaction_should,strangers_reaction,strangers_reaction_should,human_id
0,1,William did not tell someone's girlfriend that...,William,male,human,Verbal confrontation,Gossip,Do nothing,Do nothing,Do nothing,Do nothing,R_1bHJcpADL6ZNNEJ
1,1,Maria did not tell someone's girlfriend that h...,Maria,female,human,Support,Do nothing,Support,Do nothing,Do nothing,Do nothing,R_7JCKEF65rh5eA8Y
2,1,William did not tell someone's girlfriend that...,William,male,human,Verbal confrontation,Verbal confrontation,Do nothing,Do nothing,Stay away,Do nothing,R_3R48rDclzghStqN
3,1,William did not tell someone's girlfriend that...,William,male,human,Stay away,Stay away,Stay away,Stay away,Stay away,Stay away,R_7o9lY3vQcqsE4pP
4,1,William did not tell someone's girlfriend that...,William,male,human,Support,Support,Support,Support,Support,Support,R_6r37TI1xGGUVeaA
...,...,...,...,...,...,...,...,...,...,...,...,...
7726,449,Richard reads things in a British accent even ...,Richard,male,llama_pt,Gossip,Stay away,Gossip,Stay away,Gossip,Stay away,NaN
7727,449,Jennifer reads things in a British accent even...,Jennifer,female,llama_pt,Gossip,Stay away,Gossip,Stay away,Gossip,Stay away,NaN
7728,450,Charles touched a passed-out girl inappropriat...,Charles,male,llama_pt,Inform authority,Inform authority,Inform authority,Inform authority,Inform authority,Inform authority,NaN
7729,450,Elizabeth touched a passed-out guy inappropria...,Elizabeth,female,llama_pt,Stay away,Stay away,Stay away,Stay away,Inform authority,Do nothing,NaN


In [ ]:
df_base_exp['label_type'].value_counts()

,count
label_type,
human,2613
gemma_ft,853
gemma_ft_bf16,853
llama_ft,853
llama_ft_bf16,853
gemma_pt,853
llama_pt,853


In [ ]:
df_base_exp.columns

Index(['ID', 'stimuli', 'actor', 'actor_gender', 'label_type',
       'friends_reaction', 'friends_reaction_should', 'acquaintances_reaction',
       'acquaintances_reaction_should', 'strangers_reaction',
       'strangers_reaction_should', 'human_id'],
      dtype='object')

In [ ]:
df_base_exp = df_base_exp[['ID', 'stimuli', 'actor', 'actor_gender', 'label_type',
       'friends_reaction', 'friends_reaction_should', 'acquaintances_reaction',
       'acquaintances_reaction_should', 'strangers_reaction',
       'strangers_reaction_should', 'human_id']]

df_base_exp = df_base_exp.rename(columns={
    'ID': 'stimuli_id',
    'label_type': 'rater_type',
    'friends_reaction': 'friends_reaction_would',
    'acquaintances_reaction': 'acquaintances_reaction_would',
    'strangers_reaction': 'strangers_reaction_would',
})

In [ ]:
df_base_exp.to_csv(folder_path+'gemma_llama_base_exp.csv', index=False)

In [ ]:
df_intermediate = df_final.copy()

df_intermediate = df_intermediate.rename(columns={
    'stimuli_id': 'ID',
    'rater_type': 'label_type',
    'friends_reaction_would': 'friends_reaction',
    'acquaintances_reaction_would': 'acquaintances_reaction',
    'strangers_reaction_would': 'strangers_reaction',
})

df_gpt = df_intermediate.loc[df_intermediate['label_type'] == 'gpt_5.2'].drop_duplicates(["ID", "label_type", "actor_gender"]).reset_index(drop=True)
df_gpt = df_gpt[['ID', 'stimuli', 'actor', 'actor_gender', 'label_type',
       'friends_reaction_should', 'acquaintances_reaction_should', 'strangers_reaction_should', 'human_id']]
df_gpt['label_type'] = 'gpt_5.2'

In [ ]:
df_intermediate = df_neutral_exp.copy()

df_intermediate = df_intermediate.rename(columns={
    'stimuli_id': 'ID',
    'rater_type': 'label_type',
    'friends_reaction_would': 'friends_reaction',
    'acquaintances_reaction_would': 'acquaintances_reaction',
    'strangers_reaction_would': 'strangers_reaction',
})

df_gpt_neutral = df_intermediate.loc[df_intermediate['label_type'] == 'gpt_5.2_neutral'].drop_duplicates(["ID", "label_type", "actor_gender"]).reset_index(drop=True)
df_gpt_neutral = df_gpt_neutral[['ID', 'stimuli', 'actor', 'actor_gender', 'label_type',
       'friends_reaction_should', 'acquaintances_reaction_should', 'strangers_reaction_should', 'human_id']]
df_gpt_neutral['label_type'] = 'gpt_5.2_neutral'

In [ ]:
df_human_actions.columns

Index(['ID', 'stimuli', 'actor', 'actor_gender', 'label_type',
       'friends_reaction', 'friends_reaction_should', 'acquaintances_reaction',
       'acquaintances_reaction_should', 'strangers_reaction',
       'strangers_reaction_should', 'human_id'],
      dtype='object')

In [ ]:
df_human_actions_should = df_human_actions[['ID', 'stimuli', 'actor', 'actor_gender', 'label_type', 'friends_reaction_should',
       'acquaintances_reaction_should', 'strangers_reaction_should', 'human_id']]

In [ ]:
df_gpt_advice = pd.concat([df_human_actions_should, df_gpt, df_gpt_neutral, df_gpt_5_advice], ignore_index=True)
df_gpt_advice

,ID,stimuli,actor,actor_gender,label_type,friends_reaction_should,acquaintances_reaction_should,strangers_reaction_should,human_id
0,1,William did not tell someone's girlfriend that...,William,male,human,Gossip,Do nothing,Do nothing,R_1bHJcpADL6ZNNEJ
1,1,Maria did not tell someone's girlfriend that h...,Maria,female,human,Do nothing,Do nothing,Do nothing,R_7JCKEF65rh5eA8Y
2,1,William did not tell someone's girlfriend that...,William,male,human,Verbal confrontation,Do nothing,Do nothing,R_3R48rDclzghStqN
3,1,William did not tell someone's girlfriend that...,William,male,human,Stay away,Stay away,Stay away,R_7o9lY3vQcqsE4pP
4,1,William did not tell someone's girlfriend that...,William,male,human,Support,Support,Support,R_6r37TI1xGGUVeaA
...,...,...,...,...,...,...,...,...,...
5167,449,Jennifer reads things in a British accent even...,NaN,female,gpt_5.2_advice,Verbal confrontation,Do nothing,Do nothing,NaN
5168,449,Jennifer reads things in a British accent even...,NaN,male,gpt_5.2_advice,Verbal confrontation,Do nothing,Do nothing,NaN
5169,450,Elizabeth touched a passed-out guy inappropria...,NaN,male,gpt_5.2_advice,Inform authority,Inform authority,Inform authority,NaN
5170,450,Elizabeth touched a passed-out guy inappropria...,NaN,female,gpt_5.2_advice,Inform authority,Inform authority,Inform authority,NaN


In [ ]:
df_gpt_advice['label_type'].value_counts()

,count
label_type,
human,2613
gpt_5.2,853
gpt_5.2_neutral,853
gpt_5.2_advice,853


In [ ]:
df_gpt_advice.columns

Index(['ID', 'stimuli', 'actor', 'actor_gender', 'label_type',
       'friends_reaction_should', 'acquaintances_reaction_should',
       'strangers_reaction_should', 'human_id'],
      dtype='object')

In [ ]:
df_gpt_advice = df_gpt_advice[['ID', 'stimuli', 'actor', 'actor_gender', 'label_type',
       'friends_reaction_should', 'acquaintances_reaction_should',
       'strangers_reaction_should', 'human_id']]

df_gpt_advice = df_gpt_advice.rename(columns={
    'ID': 'stimuli_id',
    'label_type': 'rater_type',
})

In [ ]:
df_gpt_advice.to_csv(folder_path+'gpt_advice_neutral_exp.csv', index=False)